# Resona PRD  — Addendum: Model Family, Voices, Cloning and Ultra

Status: draft, to be merged into PRD v0.2. Same rule as before: **(measured)**, **(target)**, **(estimate)**.
Sections 18-24 are new. Sections 1, 3, 6 and 17 of v0.2 are updated where noted.

---

## 18. Resona model family (replaces the single-model )

Resona is now a **family of 3 products + 1 research model**.

| | **Resona-Basic** | **Resona-Mid** | **Resona-High** | **Resona-Ultra** (research) |
|---|---|---|---|---|
| Goal | Plug-and-play, offline, very light | Wide coverage, many voices | Studio-grade, expert level (ElevenLabs-class) | Expressive, script-aware, frontier |
| Languages | 3-4 (Hindi, Hinglish, English + 1) | 5-6 | Hindi, Hinglish, English first, then more | Same as High |
| Voices | 10 curated, best quality | 20+ | Limited flagship set + cloning | Flagship set + cloning + voice design |
| Cloning | No | No (or fixed voices only) | Yes (Section 21) | Yes, with style and emotion transfer |
| Emotion / non-verbal | Speed, pause, variation only | Basic emotion tags (neutral, happy, sad, angry) | Emotion tags + intensity | Script-driven emotion, non-verbal sounds (Section 20) |
| Runs on | CPU laptop, 8 GB, no GPU, no internet | CPU (fast) or GPU | GPU recommended, CPU possible with quantization | GPU server / high-end device |
| Size (estimate) | 50-120 MB (Lite/Base) | 300-600 MB | 1-3 GB | to be decided |
| Architecture | VITS family (current pilot) | VITS or two-stage, larger | Large model, stronger acoustic + codec-based design (to be chosen after research) | LLM-style token model or diffusion/flow with an emotion planner |
| Training data | Podcast data, curated, high quality | Larger multilingual data | Very large, expert-cleaned data (thousands of hours, target) | Largest, plus labeled emotion and non-verbal data |
| Training compute | Kaggle T4 (fits) | Needs multi-GPU rental | Needs a serious GPU cluster | Same, larger |

The current Vikram pilot becomes the first checkpoint of **Resona-Basic**. Resona-Base / Resona-Lite (Section 7.3) become the two size options inside Basic.

### 18.1 Resona-Basic (plug and play)

- 3-4 main languages, 10 best voices, small and heavily optimized. Quality first, but size and speed are hard constraints.
- Optimization plan: fp16 and int8 export, ONNX graph optimization, pruning of unused layers, smaller vocoder option, knowledge distillation **from our own larger Resona models only** (never from other companies' models), 22.05 kHz option (open decision 5).
- Targets: Section 10 gates (RTF 0.3 or lower for Base, 0.15 or lower for Lite, 1 GB RAM, 2 s startup).
- One-file install, one-line Python API, no internet. Voices bundled or downloadable.

### 18.2 Resona-Mid

- 5-6 languages (Hindi, Hinglish, English plus 2-3 Indian languages to be chosen by data availability) and 20+ voices.
- Needs more data per language; each new language needs its own text frontend and gold test set.

### 18.3 Resona-High

- Trained on far more data than the others (target: thousands of hours; needs a data plan in Section 22).
- Includes the cloning engine (Section 21), emotion tags, and fine control.
- Quality bar: MOS 4.3+ (target), measured against human recordings and named outside systems in blind tests. "ElevenLabs level" is a goal to be proven by listener tests, not a claim.

---

## 19. Voice plan (updates Section 6)

- We will have a **limited number of best-quality voices**, best in Hindi, English and Hinglish. Fewer voices, each polished, beats many average ones.
- Order of work: (1) build the best voices on Basic, (2) build the next versions (Mid, High), (3) add the remaining voices on top.
- Every voice keeps the same rules: written consent, provenance record, no imitation of real people.
- Voice tiers: Tier A flagship (High and Ultra get these), Tier B (Mid), Tier C (Basic 10-voice set).

---

## 20. Resona-Ultra: expressive model

Goal: a model that reads a script and **decides how to say it**, with human-like non-speech sounds.

### 20.1 Capabilities (target)

1. **Script-driven emotion.** The input is plain script text. A planner reads the whole script and assigns emotion, intensity and delivery to each sentence and phrase (angry, happy, sad, worried, sarcastic, tender, excited, tired...). No manual tags needed, but manual tags always override the planner.
2. **Non-verbal sounds.** Cough, throat clearing (khara-khara), "hmm", "huh", "haan", "uff", sigh, laugh, chuckle, sob, gasp, breath, sneeze, yawn, whisper, shout. Written inline as tags, for example `[hmm]`, `[cough]`, `[sigh]`, or produced automatically by the planner when the script implies them.
3. **Emotion trajectory.** Emotion can change inside a paragraph (calm rising to angry), not only per sentence.
4. **Intensity slider.** Emotion strength from 0 to 1, independent of speaker identity.
5. **Emphasis control.** Per-word stress and pause control.
6. **Hinglish-aware prosody.** Correct rhythm when Hindi and English switch inside one sentence.

### 20.2 How it works (design direction)

```
Script
  -> Script planner (small language model): emotion, intensity,
     non-verbal events, emphasis, pauses  ->  annotated script
  -> Ultra acoustic model: text + annotations + speaker/voice -> audio
  -> Optional editor: fix one word or phrase without redoing the rest
```

- The planner is trained on annotated scripts. The acoustic model is trained on speech with emotion labels and non-verbal event labels.
- Non-verbal sounds are treated as **real tokens in the vocabulary**, with their own recorded and labeled data, not as noise added later.

### 20.3 Data needed (biggest risk)

| Data | Why | How |
|---|---|---|
| Emotional speech in Hindi / Hinglish / English | Emotion control | Recorded with consenting actors, plus labeled podcast segments |
| Labeled non-verbal events | Cough, hmm, laugh, sigh etc. | Actor sessions with scripts that request each event; event tagging of podcast audio |
| Same sentence in many emotions | Lets the model separate what is said from how | Actor recording sessions (same script, 6-10 emotions) |
| Script-to-emotion labels | Train the planner | Human annotation of scripts, then model-assisted labeling with human review |

Data must follow the same provenance and consent rules (Appendix B, extended with `emotion_labels` and `nonverbal_labels`).

### 20.4 Ultra evaluation (targets)

| Metric | Target |
|---|---|
| Emotion recognition by listeners (blind) | 80% or higher match to the intended emotion |
| Non-verbal naturalness (MOS) | 3.8 or higher |
| Planner agreement with human annotators | To be measured |
| Intelligibility (WER) | No worse than neutral speech by more than 2 points |
| Identity consistency across emotions | Speaker verification score stays above threshold |

---

## 21. Voice cloning engine

**Goal:** the strongest part of Resona. Clone many types of voices from a short sample, at a level comparable to leading commercial systems.

### 21.1 What it takes (updates Section 6.3)

- A **speaker encoder** that turns any audio clip into a voice vector, feeding the model's conditioning input (the current `emb_g` lookup table cannot do this).
- Training on **hundreds to thousands of speakers**, not 20.
- Two modes:
  - **Instant cloning:** 5-30 seconds of audio, no training, runs on the fly.
  - **Pro cloning:** 30+ minutes of audio, short fine-tune for higher fidelity.
- Voice types to cover: male, female, child, elderly, deep, raspy, soft, high-pitched, regional accents, professional and casual styles. Coverage depends on the diversity of speakers in the training data.

### 21.2 Safety (required, not optional)

- **Consent check:** the person cloned must confirm consent (live phrase verification for the user's own voice).
- **Watermark:** inaudible watermark in all synthetic audio from Resona-High and Ultra, plus an open detector.
- **Block list:** no cloning of public figures or people who have not consented.
- Usage logging options for API deployments, and an acceptable-use policy in the model card.

### 21.3 Roadmap

| Step | What |
|---|---|
| C0 | Build multi-speaker data pool (100+ consented speakers) |
| C1 | Speaker encoder and conditioning path on Mid-size model |
| C2 | Instant cloning evaluation: speaker similarity and WER |
| C3 | Pro cloning (fine-tune path) |
| C4 | Safety layer: consent flow, watermark, detector |

---

## 22. Data and compute reality check

| Product | Compute | Data | Notes |
|---|---|---|---|
| Basic | Kaggle T4 is enough | 1-10 h per voice | Current path |
| Mid | Rented multi-GPU, weeks | Hundreds of hours, multi-language | Kaggle quota is not enough |
| High | GPU cluster, weeks to months | Thousands of hours (target) | Needs funding and a data-licensing plan |
| Ultra | Same as High or more | Largest, plus labeled emotion and non-verbal data | Research risk is high; build in stages |

Rule: **do not skip Basic.** It gives us the pipeline, the frontend, the evaluation tools and the first real users. Mid, High and Ultra reuse all of it.

---

## 23. Frontier features for Ultra (not standard in today's voice models)

Ideas to consider. Each needs research and measurement before being promised.

1. **Script-aware emotion planner** that reads a whole script and directs the performance (Section 20).
2. **Identity and accent separated.** Same voice can speak with a Delhi, Mumbai, Punjabi, Rajasthani, Bhojpuri or neutral accent, and a Hindi-native voice can speak English keeping identity.
3. **Voice condition controls.** Tired, hoarse, cold, out of breath, sleepy, just-woke-up, speaking while walking, phone-call quality, distance from microphone, room acoustics.
4. **Voice design from a text description.** "35-year-old, warm, slightly raspy, Delhi male" creates a new synthetic voice that belongs to no real person. Interpolate between voices.
5. **Speech editing like text.** Change one word in an audio file and keep the rest, emotion and breath included.
6. **Multi-speaker dialogue in one pass.** Natural interruptions, overlap, backchannels ("haan", "hmm") and turn-taking; auto-casting different characters in a story.
7. **Listener behavior.** For voice assistants: the model produces "listening sounds" while the user is speaking, and adapts when interrupted.
8. **Hindi-specific art forms.** Shayari, ghazal, kavita and dohe with correct rhythm and pause, plus proper nukta and Urdu-origin word pronunciation.
9. **Long-form consistency.** A full audiobook with stable character voices and emotion arcs across chapters.
10. **Learning from listener feedback.** Preference data from Hindi and Hinglish listeners to improve naturalness (feedback-based training).
11. **Built-in provenance.** Watermark and detector shipped with the model so synthetic audio can be identified.
12. **Real-time streaming with emotion.** First audio in under 300 ms (target) on a good GPU, expressive not flat.

---

## 24. Updated roadmap and risks

### Roadmap

| Phase | Content |
|---|---|
| 1 | Finish Vikram pilot (Basic), evaluation, ONNX export |
| 2 | Best 2 voices in Basic (1 male, 1 female), then 10 voices |
| 3 | Data collection for Mid and High: podcasts, actors, consent records |
| 4 | Mid: multi-language, 20+ voices, emotion tags |
| 5 | High: large-scale training, cloning engine, watermark |
| 6 | Ultra: planner, non-verbal tokens, emotion trajectory, frontier features |

### New risks

| Risk | Impact | Mitigation |
|---|---|---|
| High and Ultra need far more compute and data than we have | High | Stage the work; fund compute; do not promise dates until Basic is measured |
| Cloning misuse | High | Consent check, watermark, block list, acceptable-use policy |
| No labeled data for emotion and non-verbal sounds | High | Actor recording sessions with consent; start with 5-6 emotions and 5 non-verbal sounds |
| Claims of "ElevenLabs level" without proof | Medium | Publish blind listening tests; state results as measured |
| Trying to build all four at once | High | One product at a time; Basic ships first |

### Updated open decisions

8. Which 2-3 additional languages for Mid?
9. Architecture for High and Ultra (large VITS, codec language model, or flow/diffusion)?
10. Actor recording budget and studio plan for emotion and non-verbal data.
11. Compute plan for High and Ultra (rent, grant, partner).
12. Cloning safety policy details (consent flow and who can access it).

# Resona — Implementation Plan (Phase 1 to 6)

Yeh plan PRD v0.2 + v0.3 addendum par based hai. Rule: **ek time pe ek phase.** Har phase ka exit gate pass hone ke baad hi agla shuru hoga.

Timings are (estimate). Numbers jo measured hain PRD mein hain, unhe wahin se lena.

---

## 0. Kahan se shuruwat? (Start here)

Order of first actions, is hi order mein:

1. **Naya clean Kaggle notebook** banao (Section 1 ka structure). Purane notebook mein Whisper, F5, Parler ke objects hain jo 11 GB GPU memory kha sakte hain.
2. **Vikram ka provenance record likho** (Appendix B). Open decision #1: pilot recordings kahan se aayi, rights kya hain. Bina record ke release nahi hoga, par training pilot ke liye chal sakti hai.
3. **Pilot training launch karo** (30,000 steps, batch 4).
4. Training chalti hai us dauraan (5.5 GPU-hours), CPU side pe **gold test set** aur **podcast acquisition** start karo.

---

## 1. Notebook structure (new)

Ek notebook = ek kaam. Dependencies alag rakho (PRD 11.5).

| Notebook | Kaam | GPU? |
|---|---|---|
| `01_setup_and_data.ipynb` | Install, VITS repo patch, dataset load, filelists banana | No |
| `02_frontend.ipynb` | Hinglish to Devanagari frontend, OOV report, tests | No |
| `03_train_resona.ipynb` | Sirf training. Minimal packages. | Yes |
| `04_eval.ipynb` | Unseen-sentence samples, WER (Whisper), listening sheet | Yes (Whisper) |
| `05_export_onnx.ipynb` | ONNX export, CPU benchmark (RTF, RAM, size) | No |
| `06_podcast_pipeline.ipynb` | Demucs, diarization, Whisper transcripts, clip cutting | Some GPU |

Notebook rules (pilot lessons):
- Training se pehle **session restart**. Sirf training ke liye jo chahiye wahi load karo.
- Parler-TTS, F5, diarization jaise heavy tools **alag notebook** mein.
- `requirements.txt` pin karo.
- Checkpoints har 30 min; 5k, 10k, 20k milestone checkpoints **manually copy** karo (private HF repo ya Kaggle Dataset).
- Save Version / Save & Run All use karo, interactive GPU session idle mat chhodo.

---

## Phase 1 — Vikram pilot (Resona-Basic v0)

**Goal:** pipeline end-to-end prove karna aur pehla real model banana.
**Time:** Day 1 to 3 (training ~5.5 h + evaluation).

### Steps

| # | Step | Detail |
|---|---|---|
| 1.1 | Fresh notebook + setup | Kaggle GPU, VITS repo (`jaywalnut310/vits`) patch: Devanagari symbols, numba `monotonic_align`, `mel_processing` fix |
| 1.2 | Data load | `/kaggle/input/datasets/mishh01/voice-samples`, 404 train + 8 val clips, Hindi + Hinglish only (measured) |
| 1.3 | Frontend check | Latin guard chalao, OOV report dekho, `%` to "प्रतिशत" |
| 1.4 | Transcript review | Kam se kam 100 clips sunte hue padho (galat text = galat pronunciation) |
| 1.5 | Config | `configs/resona_base.json`; batch 4 command-line se (config mein 16 likha hai, use override karo) |
| 1.6 | Train | `python train_resona.py --model_dir /kaggle/working/runs/vikram_v1 --batch 4 --max_steps 30000 --log_every 100 --sample_every 2500 --ckpt_minutes 30 --keep 3` |
| 1.7 | Milestone listening | 5k, 10k, 20k, 30k pe fixed test sentences sunna (unseen sentences bhi) |
| 1.8 | Evaluate | Unseen-sentence test, pehla WER, overfitting check |
| 1.9 | Export + benchmark | `resona_infer.pt` to ONNX, CPU RTF / RAM / size |

### Decision rules
- Agar training clips better ho rahe hain par unseen sentences kharab: **overfitting**, purane checkpoint pe wapas jao.
- Agar out-of-memory: `segment_size` 8192 to 6144/4096, ya fp16 short test (NaN dekhna), ya Resona-Lite.

### Exit gate (Phase 1)
- [ ] 30k steps complete, milestone checkpoints saved outside `/kaggle/working`
- [ ] Voice recognizable Vikram hai
- [ ] Unseen sentences par listening report (kitne dropped/repeated words)
- [ ] ONNX file CPU pe chal rahi hai, RTF measured
- [ ] Vikram provenance record complete

**Expectation (honest):** 54 minute se scratch training = recognizable voice, par mumbling, skipped words aur weak long sentences ka risk hai. Sabse bada improvement zyada clean audio se aayega, tuning se nahi.

---

## Phase 2 — Basic ko strong banana: more data + 2 voices

**Goal:** Vikram ko better banana aur 1 female voice add karna. Phir 10 voices ki taraf.
**Time:** Week 1 to 5.

### Steps

| # | Step | Detail |
|---|---|---|
| 2.1 | Podcast acquisition | Sirf legal: open license jo training + open weight release allow kare, ya written permission. Har episode ka provenance record |
| 2.2 | Podcast pipeline (`06_`) | PRD 5.3 ke 12 steps: 24 kHz mono, Demucs, diarization, single-speaker only, speaker clustering, 2.5-12 s clips, Whisper large-v3 + second pass, manual review, frontend, quality filter, splits, manifest |
| 2.3 | Measure yield | Pehle podcast pe dekho: raw hours per clean hour (estimate 2-4 h). Us se baaki plan adjust karo |
| 2.4 | Vikram v2 | Zyada clean audio (target 3+ h) se retrain ya continue |
| 2.5 | Voice 2 (female) | Pilot checkpoint se continue, reserved speaker slot 1 use karke. Target 1+ h clean |
| 2.6 | Gold test set | 200-300 sentences: Hindi, English, code-switch, numbers, tech words, short commands, long paragraphs |
| 2.7 | Frontend v1 | Context words (`main`, `to`, `me`), spelling variants, accuracy publish |
| 2.8 | Optimization | fp16, int8 test, 22.05 kHz decision (open decision 5) |

### Exit gate (Phase 2)
- [ ] 2 voices (1 male, 1 female) at 1+ h each, consent + provenance complete
- [ ] WER on gold set measured
- [ ] RTF 0.3 or lower on 4-core CPU (Base)
- [ ] Blind pair test: voices distinct

---

## Phase 3 — Basic ko complete karna (10 voices) + Basic release

**Goal:** Resona-Basic v1.0 release: 3-4 languages, 10 best voices, lightweight.
**Time:** Week 5 to 12.

### Steps

| # | Step | Detail |
|---|---|---|
| 3.1 | Voices 3 to 10 | Ek ek voice add karo (reserved slots). Har voice ke liye consent, provenance, clean audio |
| 3.2 | 3rd/4th language decision | Data availability se decide (open decision 8) |
| 3.3 | Frontend upgrade | G2P / IPA phonemes ka decision (open decision 4); schwa deletion aur English loanwords |
| 3.4 | Full optimization | ONNX graph optimization, int8, pruning, size gate: Base 120 MB fp32 / 60 MB fp16 |
| 3.5 | Full evaluation | MOS (20+ listeners), WER, robustness 98%+, RAM 1 GB or lower, startup 2 s or lower |
| 3.6 | Packaging | Python API, CLI, sample audio, `NOTICE`, model card, acceptable-use policy |
| 3.7 | Release | Code Apache-2.0 (target), weights license per source permissions |

### Exit gate (Phase 3)
- [ ] Sab Section 10 gates (PRD v0.2) pass
- [ ] Release checklist done (licenses, consent, model card)

---

## Phase 4 — Resona-Mid

**Goal:** 5-6 languages, 20+ voices, basic emotion tags.
**Needs:** rented multi-GPU (Kaggle quota kaafi nahi), hundreds of hours of multilingual data.

| # | Step |
|---|---|
| 4.1 | Language list final karo, har language ka frontend + gold set |
| 4.2 | Data plan aur compute plan (budget/grant/partner) |
| 4.3 | Bada model (Mid architecture decision) |
| 4.4 | Emotion tags: neutral, happy, sad, angry pe labeled data |
| 4.5 | **Cloning prototype:** speaker encoder + conditioning path, 100+ consented speakers |

Exit gate: Mid gates measured, cloning prototype speaker-similarity aur WER measured.

---

## Phase 5 — Resona-High

**Goal:** expert-level quality, cloning engine, watermark.
**Needs:** GPU cluster, thousands of hours (target).

| # | Step |
|---|---|
| 5.1 | Large-scale data pipeline (Phase 2 pipeline ko scale karo) |
| 5.2 | Architecture research: large VITS vs codec language model vs flow/diffusion |
| 5.3 | Instant cloning (5-30 s) + pro cloning (fine-tune) |
| 5.4 | Safety: consent check, watermark + open detector, block list |
| 5.5 | Blind listening tests vs outside systems (sirf information ke liye) |

Exit gate: MOS 4.3+ (target), cloning similarity measured, safety layer live.

---

## Phase 6 — Resona-Ultra

**Goal:** script-aware emotion, non-verbal sounds, frontier features.

| # | Step |
|---|---|
| 6.1 | **Actor recording sessions** (consent): same script, 6-10 emotions; 5 non-verbal sounds (cough, khara-khara, hmm, huh, sigh) |
| 6.2 | Non-verbal tokens vocabulary mein add karo, labeled data ke saath |
| 6.3 | Script planner (small LM): emotion, intensity, pauses, non-verbal events |
| 6.4 | Emotion trajectory + intensity slider |
| 6.5 | Frontier features ek ek karke (PRD v0.3 Section 23): accent/identity split, voice condition controls, voice design, speech editing, dialogue |

Start small: 5-6 emotions + 5 non-verbal sounds, phir expand.
Exit gate: emotion recognition 80%+ (blind), non-verbal MOS 3.8+, identity consistent across emotions.

---

## Parallel tracks (Phase 1 ke saath shuru)

| Track | Kya | Kab tak |
|---|---|---|
| Provenance | Har source ka record, consent forms | Continuous |
| Gold test set | 200-300 hand-verified sentences | Phase 2 se pehle |
| Compute/funding plan | Mid/High/Ultra ke liye | Phase 3 tak |
| Actor/studio plan | Emotion + non-verbal recording | Phase 5 tak |
| Cloning safety policy | Consent flow, watermark design | Phase 4 tak |

---

## Immediate checklist (aaj se)

- [ ] Naya notebook `03_train_resona` banao, fresh session
- [ ] Vikram provenance record likho
- [ ] Phase 1 command launch karo
- [ ] Training ke dauraan: gold test set likhna shuru karo
- [ ] Podcast sources ki list banao (legal, permission ke saath)
- [ ] 5k step pe pehle samples sunno aur notes likho

### ONE PIPELINE FOR CLEANING DATA (MULTI SPACKER ALLOWED MAX 2 ) 

In [1]:
!ls -lh /kaggle/input/datasets/mishh01/raj-data

total 670M
-rw-r--r-- 1 nobody nogroup 102M Oct  3 09:18 'Raj Sample 1.mp3'
-rw-r--r-- 1 nobody nogroup 187M Oct  3 09:18 'raj Sample 2.mp3'
-rw-r--r-- 1 nobody nogroup 100M Oct  3 09:18 'Raj sample 3.mp3'
-rw-r--r-- 1 nobody nogroup 161M Oct  3 09:18 'Raj sample 4.mp3'
-rw-r--r-- 1 nobody nogroup 122M Oct  3 09:18 'raj sample 5.mp3'


In [2]:
# =====================================================================
# RESONA DATA PREP v2  -  ONE CELL
# Long recording with 2 people  ->  each person separated automatically
#   -> cleaned -> cut into 2.5-12 s chunks -> saved in their OWN folder
#
#   OUTPUT_DIR/speakers/spk_A/chunks/*.wav   (spk_A = the one who speaks more)
#   OUTPUT_DIR/speakers/spk_B/chunks/*.wav
#   OUTPUT_DIR/speakers/spk_X/review/        random clips: LISTEN and see who is who
#   OUTPUT_DIR/rejected/<reason>/            noise, music, overlap, mixed speakers...
#   OUTPUT_DIR/manifest.csv, speaker_report.json
#
# Needs Internet ON in Kaggle (downloads a small speaker-ID model, once).
# Paste this whole cell and run. Edit only the CONFIG block.
# =====================================================================

# ------------------------------- CONFIG ------------------------------
INPUT_DIR      = "/kaggle/input/datasets/mishh01/raj-data"   # folder with the long recording(s)
OUTPUT_DIR     = "/kaggle/working/resona_data"   # test mode writes to OUTPUT_DIR + "_test" automatically
TEST_MINUTES   = 0      # >0 = TEST MODE: only the first N minutes of the first file. Set 0 for the FULL run.
PART_SECONDS   = 3600    # long files are auto-split into parts this long (keeps RAM safe)
NUM_SPEAKERS   = 6       # how many different people speak in the recordings
USE_DEMUCS     = False   # True = remove background music first (slow on 3 h audio)
DENOISE        = "off"   # "off" or "light"
MIN_SNR_DB     = 25.0    # lower (20) keeps more clips, higher (30) is stricter
STRICTNESS     = "normal"  # "normal" | "strict"  (strict = only very sure speaker clips, less data but purer)
SAVE_REJECTED  = True
REVIEW_N       = 20      # random clips per speaker copied to review/
PLAY_N         = 3       # clips per speaker played inline in the notebook
RESET          = True   # True = delete OUTPUT_DIR and start from zero
KEEP_CANDIDATES = False  # True = keep temporary _cand folder
# ---------------------------------------------------------------------

import sys, subprocess, importlib, shutil, os, re, csv, json, hashlib, random, tempfile, traceback, pickle
from collections import Counter
from pathlib import Path


def _need(mod, pip_name=None):
    try:
        return importlib.import_module(mod)
    except ImportError:
        print(f"installing {pip_name or mod} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or mod], check=False)
        importlib.invalidate_caches()
        return importlib.import_module(mod)


import numpy as np
from scipy import signal
sf = _need("soundfile")
if DENOISE == "light":
    _need("noisereduce")
if USE_DEMUCS:
    _need("demucs")

# ------------------------------ SETTINGS -----------------------------
SR, SR_EMB = 24000, 16000
AUDIO_EXT = {".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".mp4", ".mkv", ".webm", ".mov", ".wma"}
HOP_MS, WIN_HOPS = 10.0, 2
MIN_CHUNK_S, MAX_CHUNK_S = 2.5, 12.0
MIN_SILENCE_S, MAX_MERGE_GAP_S, PAD_S = 0.30, 0.50, 0.12   # smaller merge gap: fewer speaker changes inside a chunk
SPEECH_MARGIN_DB, MIN_SPEECH_S = 12.0, 0.30
MAX_CLIP_FRAC, MIN_RMS_DB, MIN_SPEECH_RATIO = 0.001, -45.0, 0.55
TARGET_RMS_DB, PEAK_LIMIT, HIGHPASS_HZ = -22.0, 0.95, 70.0
WIN_S, WIN_HOP_S = 1.6, 0.8                       # speaker-embedding windows
MIN_MARGIN = 0.10 if STRICTNESS == "normal" else 0.18   # how much closer to own speaker than to the other
MIX_FRAC = 0.20 if STRICTNESS == "normal" else 0.08     # allowed share of windows that look like the other speaker
OUTLIER_STD = 2.0 if STRICTNESS == "normal" else 1.5
HS = HOP_MS / 1000
HOP = int(SR * HS)
WIN, WIN_HOP = int(WIN_S * SR_EMB), int(WIN_HOP_S * SR_EMB)


# ------------------------------- DECODE ------------------------------
def decode(path, sr=SR):
    if shutil.which("ffmpeg"):
        r = subprocess.run(["ffmpeg", "-v", "error", "-i", str(path), "-vn", "-ac", "1", "-ar", str(sr),
                            "-f", "f32le", "-"], capture_output=True)
        if r.returncode == 0 and len(r.stdout) > 0:
            return np.nan_to_num(np.frombuffer(r.stdout, dtype=np.float32).copy())
        raise RuntimeError("ffmpeg could not read file: " + (r.stderr.decode(errors="ignore").strip().splitlines() or ["unknown error"])[0][:150])
    x, fs = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if fs != sr:
        g = int(np.gcd(fs, sr))
        x = signal.resample_poly(x, sr // g, fs // g).astype(np.float32)
    return np.nan_to_num(x)


def demucs_vocals(path):
    try:
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        with tempfile.TemporaryDirectory() as td:
            r = subprocess.run([sys.executable, "-m", "demucs", "--two-stems=vocals", "-d", dev,
                                "-o", td, str(path)], capture_output=True)
            if r.returncode != 0:
                raise RuntimeError(r.stderr.decode(errors="ignore")[-200:])
            return decode(next(Path(td).rglob("vocals.wav")))
    except Exception as ex:
        print(f"   [warn] demucs failed ({ex}); using original audio")
        return decode(path)


# ---------------------------- CLEANING ENGINE ------------------------
def clean_audio(x):
    x = x - np.mean(x)
    sos = signal.butter(4, HIGHPASS_HZ, "highpass", fs=SR, output="sos")
    x = signal.sosfiltfilt(sos, x).astype(np.float32)
    if DENOISE == "light":
        try:
            import noisereduce as nr
            blk = SR * 120
            x = np.concatenate([nr.reduce_noise(y=x[i:i + blk], sr=SR, stationary=True,
                                                prop_decrease=0.5).astype(np.float32)
                                for i in range(0, len(x), blk)])
        except Exception as ex:
            print(f"   [warn] denoise skipped: {ex}")
    return x


# ------------------------------ CHUNK MAKER --------------------------
def frame_db(x):
    n = len(x) // HOP
    if n < WIN_HOPS + 1:
        return np.full(1, -100.0)
    a = x[: n * HOP].reshape(n, HOP)
    e = np.einsum("ij,ij->i", a, a) / HOP
    e = np.convolve(e, np.ones(WIN_HOPS) / WIN_HOPS, mode="same")
    return 10 * np.log10(e + 1e-10)


def runs(mask):
    if not mask.any():
        return []
    d = np.diff(np.concatenate([[0], mask.astype(np.int8), [0]]))
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))


def find_segments(db):
    nf = float(np.percentile(db, 10))
    thr = max(nf + SPEECH_MARGIN_DB, -55.0)
    speech = db > thr
    gap = int(MIN_SILENCE_S / HS)
    for s, e in runs(~speech):
        if e - s < gap and s > 0 and e < len(speech):
            speech[s:e] = True
    return [(s, e) for s, e in runs(speech) if (e - s) * HS >= MIN_SPEECH_S], thr, nf


def make_chunks(segs, db):
    maxh, gaph, pad = int(MAX_CHUNK_S / HS), int(MAX_MERGE_GAP_S / HS), int(PAD_S / HS)
    merged = []
    for s, e in segs:
        if merged and s - merged[-1][1] <= gaph and (e - merged[-1][0] + 2 * pad) <= maxh:
            merged[-1] = (merged[-1][0], e)
        else:
            merged.append((s, e))
    out = []
    for s, e in merged:
        s, e = max(0, s - pad), min(len(db), e + pad)
        while (e - s) > maxh:
            lo = max(s + maxh - int(4 / HS), s + int(MIN_CHUNK_S / HS))
            cut = lo + int(np.argmin(db[lo:s + maxh]))
            out.append((s, cut))
            s = cut
        out.append((s, e))
    return out


def metrics(seg_db, x, thr, nf):
    return {"snr_db": round(float(np.percentile(seg_db, 95) - nf), 1),
            "rms_db": round(float(np.percentile(seg_db, 95)), 1),
            "peak": round(float(np.max(np.abs(x))), 3),
            "clip_frac": float(np.mean(np.abs(x) >= 0.99)),
            "speech_ratio": round(float((seg_db > thr).mean()), 2)}


def judge(m, dur):
    if dur < MIN_CHUNK_S:                    return "too_short"
    if m["clip_frac"] > MAX_CLIP_FRAC:       return "clipping"
    if m["rms_db"] < MIN_RMS_DB:             return "too_quiet"
    if m["snr_db"] < MIN_SNR_DB:             return "low_snr_noise_or_music"
    if m["speech_ratio"] < MIN_SPEECH_RATIO: return "mostly_silence"
    return ""


def normalize(x):
    gain = 10 ** (TARGET_RMS_DB / 20) / np.sqrt(np.mean(x ** 2) + 1e-12)
    gain = min(gain, PEAK_LIMIT / (np.max(np.abs(x)) + 1e-9))
    return np.clip(x * gain, -1.0, 1.0)


def save_wav(path, x):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x, SR, subtype="PCM_16")


# --------------------------- SPEAKER EMBEDDER ------------------------
def load_embedder(out):
    """Returns embed(batch[B, WIN] float32 16 kHz) -> (B, D). Tries SpeechBrain ECAPA, then Resemblyzer."""
    try:
        import torch
        _need("speechbrain")
        try:
            from speechbrain.inference.speaker import EncoderClassifier
        except ImportError:
            from speechbrain.pretrained import EncoderClassifier
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        clf = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                             savedir=str(out / "_spk_model"), run_opts={"device": dev})

        def embed(b):
            with torch.no_grad():
                return clf.encode_batch(torch.from_numpy(b).to(dev)).squeeze(1).cpu().numpy()
        print(f"speaker model: SpeechBrain ECAPA on {dev}")
        return embed
    except Exception as ex:
        print(f"[warn] SpeechBrain failed ({str(ex)[:120]}); trying Resemblyzer")
    try:
        _need("resemblyzer")
        from resemblyzer import VoiceEncoder
        enc = VoiceEncoder()

        def embed(b):
            return np.stack([enc.embed_utterance(w) for w in b])
        print("speaker model: Resemblyzer")
        return embed
    except Exception as ex:
        raise SystemExit("Could not load any speaker model. Turn Internet ON in Kaggle "
                         f"(Settings > Internet) and rerun.\nDetail: {str(ex)[:200]}")


def embed_all(embed, waves, bs=64):
    outs = []
    for i in range(0, len(waves), bs):
        outs.append(embed(np.stack(waves[i:i + bs]).astype(np.float32)))
    e = np.concatenate(outs) if outs else np.zeros((0, 1))
    return e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-9)


def kmeans_cos(E, k, n_init=20, iters=60, seed=0):
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_init):
        idx = [int(rng.integers(len(E)))]
        for _ in range(k - 1):
            d = 1 - np.max(E @ E[idx].T, axis=1)
            p = d ** 2 + 1e-12
            idx.append(int(rng.choice(len(E), p=p / p.sum())))
        C = E[idx].copy()
        for _ in range(iters):
            lab = np.argmax(E @ C.T, axis=1)
            newC = np.stack([E[lab == j].mean(0) if (lab == j).any() else C[j] for j in range(k)])
            newC /= np.linalg.norm(newC, axis=1, keepdims=True)
            if np.allclose(newC, C):
                break
            C = newC
        score = float(np.max(E @ C.T, axis=1).sum())
        if best is None or score > best[0]:
            best = (score, np.argmax(E @ C.T, axis=1), C)
    return best[1], best[2]


# ------------------------------ PASS 1 (per file) --------------------
def safe_name(p, root):
    stem = re.sub(r"[^A-Za-z0-9]+", "_", p.stem).strip("_")[:40] or "file"
    return f"{stem}_{hashlib.md5(str(p.relative_to(root)).encode()).hexdigest()[:4]}"


def pass1(p, root, out, embed, st):
    """Clean + chunk + quality gate + speaker embeddings. Saves candidates in _cand/."""
    name = safe_name(p, root)
    pk = out / "_cand" / f"{name}.pkl"
    if pk.exists():
        return
    x = demucs_vocals(p) if USE_DEMUCS else decode(p)
    st["hours_in"] += len(x) / SR / 3600
    if len(x) < 5 * SR:
        raise RuntimeError("audio shorter than 5 seconds")
    x = clean_audio(x)
    db = frame_db(x)
    segs, thr, nf = find_segments(db)
    recs, waves, owner = [], [], []
    for i, (s, e) in enumerate(make_chunks(segs, db)):
        seg = x[s * HOP: e * HOP]
        dur = len(seg) / SR
        m = metrics(db[s:e], seg, thr, nf)
        reason = judge(m, dur)
        cid = f"{name}_{i:04d}"
        rec = {"cid": cid, "source": str(p.relative_to(root)), "start_s": round(s * HS, 2),
               "end_s": round(e * HS, 2), "dur": round(dur, 2), "snr": m["snr_db"], "rms": m["rms_db"],
               "reason": reason, "win": None}
        seg_n = normalize(seg)
        if reason:
            st["rej_" + reason] += 1
            if SAVE_REJECTED and dur >= 0.5:
                save_wav(out / "rejected" / reason / f"{cid}.wav", seg_n)
        else:
            save_wav(out / "_cand" / "wav" / f"{cid}.wav", seg_n)
            s16 = signal.resample_poly(seg_n, 2, 3).astype(np.float32)
            n_w = 0
            for a in range(0, len(s16) - WIN + 1, WIN_HOP):
                w = s16[a:a + WIN]
                if 10 * np.log10(np.mean(w ** 2) + 1e-10) > thr + 3:   # skip pauses
                    waves.append(w)
                    owner.append(len(recs))
                    n_w += 1
            if n_w == 0:
                rec["reason"] = "no_speech_window"
                st["rej_no_speech_window"] += 1
        recs.append(rec)
    del x, db
    if waves:
        E = embed_all(embed, waves)
        by = {}
        for e_, o in zip(E, owner):
            by.setdefault(o, []).append(e_)
        for o, lst in by.items():
            recs[o]["win"] = np.stack(lst)
    with open(pk, "wb") as f:
        pickle.dump(recs, f)


# ------------------------------ PASS 2 (all files) -------------------
def pass2(out):
    recs = []
    for pk in sorted((out / "_cand").glob("*.pkl")):
        recs += pickle.load(open(pk, "rb"))
    good = [r for r in recs if not r["reason"] and r["win"] is not None]
    report = {"candidates": len(good)}
    if len(good) < 10 * NUM_SPEAKERS:
        raise SystemExit(f"Only {len(good)} usable chunks after cleaning - too few to separate speakers. "
                         "Lower MIN_SNR_DB or check the recordings.")
    # chunk embedding = mean of its speech windows
    E = np.stack([r["win"].mean(0) for r in good])
    E /= np.linalg.norm(E, axis=1, keepdims=True) + 1e-9
    lab, C = kmeans_cos(E, NUM_SPEAKERS)
    sims = E @ C.T
    # order speakers by total speech time: spk_A = speaks most
    dur_by = [sum(r["dur"] for r, l in zip(good, lab) if l == k) for k in range(NUM_SPEAKERS)]
    order = list(np.argsort(dur_by)[::-1])
    names = {int(k): "spk_" + "ABCDEFGH"[i] for i, k in enumerate(order)}
    cc = C @ C.T
    report["centroid_similarity"] = {f"{names[i]}-{names[j]}": round(float(cc[i, j]), 3)
                                     for i in range(NUM_SPEAKERS) for j in range(i + 1, NUM_SPEAKERS)}
    own = sims[np.arange(len(good)), lab]
    stats_k = {k: (own[lab == k].mean(), own[lab == k].std() + 1e-6) for k in range(NUM_SPEAKERS)}
    keep_rows, rej_rows = [], []
    for r, l, sm, o in zip(good, lab, sims, own):
        l = int(l)
        srt = np.sort(sm)[::-1]
        margin = float(srt[0] - srt[1])
        reason = ""
        if margin < MIN_MARGIN:
            reason = "uncertain_speaker"
        elif o < stats_k[l][0] - OUTLIER_STD * stats_k[l][1]:
            reason = "speaker_outlier"
        else:
            wl = np.argmax(r["win"] @ C.T, axis=1)
            if np.mean(wl != l) > MIX_FRAC:
                reason = "mixed_speakers"
        r["speaker"], r["conf"], r["final"] = names[l], round(margin, 3), reason
        (rej_rows if reason else keep_rows).append(r)

    # move files
    for r in keep_rows:
        dst = out / "speakers" / r["speaker"] / "chunks" / f"{r['cid']}.wav"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(out / "_cand" / "wav" / f"{r['cid']}.wav"), dst)
    for r in rej_rows:
        src = out / "_cand" / "wav" / f"{r['cid']}.wav"
        if SAVE_REJECTED:
            dst = out / "rejected" / r["final"] / f"{r['cid']}.wav"
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), dst)

    # manifest
    fields = ["chunk_id", "speaker", "source", "start_s", "end_s", "duration_s", "snr_db", "rms_db",
              "speaker_conf", "status", "reason", "text"]
    with open(out / "manifest.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, lineterminator="\n")
        w.writeheader()
        for r in recs:
            fin = r.get("final", None)
            status = "kept" if (not r["reason"] and fin == "") else "rejected"
            w.writerow({"chunk_id": r["cid"], "speaker": r.get("speaker", ""), "source": r["source"],
                        "start_s": r["start_s"], "end_s": r["end_s"], "duration_s": r["dur"],
                        "snr_db": r["snr"], "rms_db": r["rms"], "speaker_conf": r.get("conf", ""),
                        "status": status, "reason": r["reason"] or (fin or ""), "text": ""})
    rej_count = Counter((r["final"] or "") for r in rej_rows)
    report["speaker_rejects"] = dict(rej_count)
    report["speakers"] = {}
    for k in order:
        nm = names[int(k)]
        rows = [r for r in keep_rows if r["speaker"] == nm]
        report["speakers"][nm] = {"chunks": len(rows),
                                  "hours": round(sum(r["dur"] for r in rows) / 3600, 3)}
    return report


# ------------------------- LONG-FILE SPLITTER ------------------
def probe_duration(p):
    if not shutil.which("ffprobe"):
        return None
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of",
                        "default=nw=1:nk=1", str(p)], capture_output=True)
    try:
        return float(r.stdout.decode().strip())
    except Exception:
        return None


def prepare_inputs(files, inp, out):
    """Returns [(path, root)]. Long files (or test mode) are split into wav parts so RAM stays safe."""
    pairs, tmp = [], out / "_parts"
    for p in files:
        dur = probe_duration(p)
        limit = TEST_MINUTES * 60 if TEST_MINUTES else None
        need_split = shutil.which("ffmpeg") and (limit or (dur and dur > PART_SECONDS * 1.05))
        if not need_split:
            pairs.append((p, inp))
            continue
        d = tmp / safe_name(p, inp)
        if not (d / ".done").exists():
            shutil.rmtree(d, ignore_errors=True)
            d.mkdir(parents=True, exist_ok=True)
            print(f"splitting {p.name} ({(dur or 0)/3600:.2f} h) into {PART_SECONDS//60}-min parts ...")
            cmd = ["ffmpeg", "-v", "error", "-y", "-i", str(p), "-vn", "-ac", "1", "-ar", str(SR)]
            if limit:
                cmd += ["-t", str(limit)]
            cmd += ["-f", "segment", "-segment_time", str(PART_SECONDS), "-c:a", "pcm_s16le", str(d / "part_%02d.wav")]
            r = subprocess.run(cmd, capture_output=True)
            if r.returncode != 0:
                print("   [warn] split failed, using file as is:", r.stderr.decode(errors="ignore")[:150])
                pairs.append((p, inp))
                continue
            (d / ".done").write_text("ok")
        pairs += [(q, tmp) for q in sorted(d.glob("*.wav"))]
        if limit:
            break          # test mode: first file only
    return pairs


# -------------------------------- DRIVER -----------------------------
def main():
    inp = Path(INPUT_DIR)
    out = Path(OUTPUT_DIR + ("_test" if TEST_MINUTES else ""))
    if not inp.exists():
        found = [str(d) for d in Path("/kaggle/input").glob("*")] if Path("/kaggle/input").exists() else []
        raise SystemExit(f"INPUT_DIR not found: {inp}\nAvailable input folders: {found or 'none'}")
    files = sorted(p for p in inp.rglob("*") if p.suffix.lower() in AUDIO_EXT)
    if not files:
        raise SystemExit(f"No audio/video files found in {inp}")
    if RESET and out.exists():
        shutil.rmtree(out)
    for d in ("_cand/wav", "speakers"):
        (out / d).mkdir(parents=True, exist_ok=True)
    work = prepare_inputs(files, inp, out)
    print(("TEST MODE: " if TEST_MINUTES else "FULL RUN: ") + f"{len(work)} part(s) to process | output: {out}")
    print(f"{len(files)} source file(s) | speakers={NUM_SPEAKERS} demucs={USE_DEMUCS} denoise={DENOISE} "
          f"min_snr={MIN_SNR_DB} strictness={STRICTNESS}")

    embed = load_embedder(out)
    st, failed = Counter(), []
    print("\n--- Step 1/2: cleaning, chunking, speaker fingerprints ---")
    for n, (p, root) in enumerate(work, 1):
        print(f"[{n}/{len(work)}] {p.relative_to(root)}")
        try:
            pass1(p, root, out, embed, st)
        except Exception as ex:
            print(f"   [skipped] {ex}")
            failed.append(f"{p.relative_to(root)}\t{ex}")
    if failed:
        (out / "failed_files.txt").write_text("\n".join(failed))

    print("\n--- Step 2/2: separating speakers ---")
    rep = pass2(out)
    rep["failed_files"] = len(failed)
    rep["quality_rejects"] = {k[4:]: v for k, v in st.items() if k.startswith("rej_")}
    (out / "speaker_report.json").write_text(json.dumps(rep, indent=2))

    print("\n================ SUMMARY ================")
    print(json.dumps(rep, indent=2))
    sim = list(rep["centroid_similarity"].values())
    if sim and max(sim) > 0.80:
        print("\n[WARNING] The two speaker voices look very similar to the model (or only one person speaks).")
        print("          Listen to review/ folders carefully before using this data.")
    for nm, s in rep["speakers"].items():
        if s["hours"] < 1.0:
            print(f"[note] {nm}: only {s['hours']:.2f} h clean speech (aim for 1+ h).")

    for sp in sorted((out / "speakers").glob("spk_*")):
        ch = sorted((sp / "chunks").glob("*.wav"))
        if REVIEW_N and ch:
            rv = sp / "review"
            shutil.rmtree(rv, ignore_errors=True)
            rv.mkdir()
            for k in random.sample(ch, min(REVIEW_N, len(ch))):
                shutil.copy(k, rv / k.name)
    print("\nNEXT: listen to speakers/spk_A/review and spk_B/review. Check that each folder has ONE person.")
    print("Then rename folders, e.g.  os.rename('.../speakers/spk_A', '.../speakers/vikram')")
    try:
        from IPython.display import Audio, display
        for sp in sorted((out / "speakers").glob("spk_*")):
            ch = sorted((sp / "chunks").glob("*.wav"))
            for k in random.sample(ch, min(PLAY_N, len(ch))):
                print(sp.name, k.name)
                display(Audio(str(k)))
    except Exception:
        pass
    if not KEEP_CANDIDATES:
        shutil.rmtree(out / "_cand", ignore_errors=True)
        shutil.rmtree(out / "_parts", ignore_errors=True)
    if TEST_MINUTES:
        print("\nThis was a TEST. If the speaker folders sound right, set TEST_MINUTES = 0 and run the full data.")


try:
    main()
except SystemExit as e:
    print(e)
except Exception:
    print("UNEXPECTED ERROR (send this to me):")
    traceback.print_exc()

splitting Raj Sample 1.mp3 (1.84 h) into 60-min parts ...
splitting Raj sample 3.mp3 (1.82 h) into 60-min parts ...
splitting Raj sample 4.mp3 (2.92 h) into 60-min parts ...
splitting raj Sample 2.mp3 (3.40 h) into 60-min parts ...
splitting raj sample 5.mp3 (2.21 h) into 60-min parts ...
FULL RUN: 14 part(s) to process | output: /kaggle/working/resona_data
5 source file(s) | speakers=6 demucs=False denoise=off min_snr=25.0 strictness=normal
installing speechbrain ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 38.1 MB/s eta 0:00:00


hyperparams.yaml: 0.00B [00:00, ?B/s]

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

...

spk_E part_00_5d77_0291.wav


spk_E part_00_5d77_0405.wav


spk_F part_00_58a3_0389.wav


spk_F part_00_58a3_0246.wav


spk_F part_01_67cf_0405.wav


In [3]:
# =====================================================================
# RESONA - NAME THE VOICES  (one cell)
# spk_A ... spk_F  ->  arjun, neha, ...   (gender is detected from pitch automatically)
# Safe to run twice. Creates voices.json and updates manifest.csv.
# =====================================================================
import os, json, random
import numpy as np, pandas as pd, soundfile as sf
from pathlib import Path

BASE = Path("/kaggle/working/resona_data")

# Optional: force a gender if the automatic detection is wrong, e.g. {"spk_C": "f"}
OVERRIDE = {}

MALE   = ["arjun", "kabir", "rohan", "vivaan", "aditya", "yash", "dev", "nikhil"]
FEMALE = ["neha", "kavya", "riya", "ananya", "isha", "meera", "pooja", "sneha"]


def median_f0(files, n=25, sr=24000):
    """Median pitch (Hz) of voiced frames over n random clips. Simple autocorrelation, numpy only."""
    files = random.Random(0).sample(files, min(n, len(files)))
    f0s, fl, hop = [], 1024, 480
    lo, hi = int(sr / 400), int(sr / 70)
    for f in files:
        x, _ = sf.read(str(f), dtype="float32")
        x = x.mean(axis=1) if x.ndim > 1 else x
        if len(x) < fl * 2:
            continue
        idx = np.arange(0, len(x) - fl, hop)
        fr = np.stack([x[i:i + fl] for i in idx]) * np.hanning(fl)
        e = (fr ** 2).mean(1)
        fr = fr[e > np.percentile(e, 50)]                       # louder half = voiced speech
        ac = np.fft.irfft(np.abs(np.fft.rfft(fr, 2048, axis=1)) ** 2, axis=1)[:, :hi + 1]
        ac /= ac[:, :1] + 1e-9
        seg = ac[:, lo:hi]
        peak = seg.max(1)
        first = (seg >= 0.9 * peak[:, None]).argmax(1)          # first strong peak = avoids octave errors
        voiced = peak > 0.5
        f0s += list(sr / (first[voiced] + lo))
    return float(np.median(f0s)) if f0s else None


spk_dirs = sorted(d for d in (BASE / "speakers").glob("spk_*") if d.is_dir())
if not spk_dirs:
    print("No spk_* folders found - already renamed? Current registry:")
    print(pd.DataFrame(json.loads((BASE / "voices.json").read_text()))[["slot", "name", "gender", "chunks", "hours"]])
else:
    rows = []
    for d in spk_dirs:
        files = sorted((d / "chunks").glob("*.wav"))
        f0 = median_f0(files)
        g = OVERRIDE.get(d.name) or (None if f0 is None else ("m" if f0 < 165 else "f"))
        rows.append({"old": d.name, "f0": f0, "g": g or "m", "check": (f0 is None) or (145 <= f0 <= 185)})

    male, female, NAMES = list(MALE), list(FEMALE), {}
    for r in rows:
        NAMES[r["old"]] = (male if r["g"] == "m" else female).pop(0)

    registry = []
    for slot, r in enumerate(rows):
        src, dst = BASE / "speakers" / r["old"], BASE / "speakers" / NAMES[r["old"]]
        os.rename(src, dst)
        files = list((dst / "chunks").glob("*.wav"))
        hours = sum(sf.info(str(f)).duration for f in files) / 3600
        registry.append({"name": NAMES[r["old"]], "slot": slot, "gender": "male" if r["g"] == "m" else "female",
                         "median_pitch_hz": None if r["f0"] is None else round(r["f0"]),
                         "folder": str(dst), "chunks": len(files), "hours": round(hours, 3),
                         "style": "", "consent": "pending", "source": "raj-data podcasts"})

    m = pd.read_csv(BASE / "manifest.csv")
    m["speaker"] = m["speaker"].replace(NAMES)
    m.to_csv(BASE / "manifest.csv", index=False)
    (BASE / "voices.json").write_text(json.dumps(registry, indent=2))

    print(pd.DataFrame(registry)[["slot", "name", "gender", "median_pitch_hz", "chunks", "hours"]].to_string(index=False))
    for r in rows:
        if r["check"]:
            print(f"[check by ear] {NAMES[r['old']]}: pitch is near the male/female boundary. "
                  f"Listen to a few clips; if wrong, rename the folder and edit voices.json.")
    print("\nMale usually ~85-155 Hz, female ~165-255 Hz. Confirm by listening to 2-3 clips per voice.")

 slot  name gender  median_pitch_hz  chunks  hours
    0 arjun   male              142     994  2.059
    1  neha female              203     970  1.957
    2 kabir   male              133     627  1.223
    3 rohan   male              133     664  1.432
    4 kavya female              167     601  1.305
    5  riya female              209     468  0.971
[check by ear] kavya: pitch is near the male/female boundary. Listen to a few clips; if wrong, rename the folder and edit voices.json.

Male usually ~85-155 Hz, female ~165-255 Hz. Confirm by listening to 2-3 clips per voice.


In [4]:
import os, json, pandas as pd
from pathlib import Path

BASE = Path("/kaggle/working/resona_data")
OLD, NEW, GENDER = "kavya", "vivaan", "male"

reg_path = BASE / "voices.json"
reg = json.loads(reg_path.read_text())
if any(v["name"] == NEW for v in reg):
    print(f"{NEW} already exists - nothing to do.")
else:
    src, dst = BASE / "speakers" / OLD, BASE / "speakers" / NEW
    assert src.exists(), f"folder not found: {src}"
    os.rename(src, dst)
    for v in reg:
        if v["name"] == OLD:
            v.update(name=NEW, gender=GENDER, folder=str(dst))
    reg_path.write_text(json.dumps(reg, indent=2))
    m = pd.read_csv(BASE / "manifest.csv")
    m["speaker"] = m["speaker"].replace({OLD: NEW})
    m.to_csv(BASE / "manifest.csv", index=False)
print(pd.DataFrame(reg)[["slot", "name", "gender", "median_pitch_hz", "chunks", "hours"]].to_string(index=False))

 slot   name gender  median_pitch_hz  chunks  hours
    0  arjun   male              142     994  2.059
    1   neha female              203     970  1.957
    2  kabir   male              133     627  1.223
    3  rohan   male              133     664  1.432
    4 vivaan   male              167     601  1.305
    5   riya female              209     468  0.971


In [5]:
# =====================================================================
# RESONA TEXT FRONTEND v1  -  Hinglish -> natural mixed script  (ONE CELL)
#   Hindi words   -> Devanagari      (mera -> मेरा, hai -> है)
#   English words -> stay in English (meeting, basically, office)
#   Names         -> stay as written (Milan)
# Same rule for Roman input, Devanagari input (Whisper output) and mixed input.
#
#   convert("Hello mera naam Milan hai aur isko basically...")
#     -> "Hello मेरा नाम Milan है और इसको basically..."
#
# Uses: (1) a Hindi-roman word list, (2) context for ambiguous words (main / to / is / me),
#       (3) a spelling-pattern guess + rule transliteration for unknown Hindi-looking words.
# Every unknown word is logged in an OOV report so you can fix it (add to EXTRA_HINDI / EXTRA_ENGLISH).
# =====================================================================

# ------------------------------- CONFIG ------------------------------
RUN_ON_TRANSCRIPTS = False   # True = read BASE/transcripts.jsonl, write transcripts_norm.jsonl + oov_report.csv
BASE               = "/kaggle/working/resona_data"
DEV_LOAN_TO_LATIN  = True    # Devanagari loanwords from Whisper (मीटिंग) -> Latin (meeting)
EXTRA_HINDI   = {}           # force: {"sarkar": "सरकार"}
EXTRA_ENGLISH = set()        # force keep-as-English: {"milan", "kotlin"}
# ---------------------------------------------------------------------

import re, json, csv
from collections import Counter
from pathlib import Path


def _pairs(s):
    return dict(p.split("=", 1) for p in s.split())


# ------------------------- HINDI (roman -> Devanagari) ---------------
HI = _pairs("""
mera=मेरा meri=मेरी mere=मेरे tera=तेरा teri=तेरी tere=तेरे tumhara=तुम्हारा tumhari=तुम्हारी tumhare=तुम्हारे
hamara=हमारा hamari=हमारी hamare=हमारे aapka=आपका aapki=आपकी aapke=आपके apna=अपना apni=अपनी apne=अपने
uska=उसका uski=उसकी uske=उसके iska=इसका iski=इसकी iske=इसके unka=उनका unki=उनकी unke=उनके
naam=नाम hai=है hain=हैं hu=हूँ hun=हूँ hoon=हूँ ho=हो hoga=होगा hogi=होगी honge=होंगे tha=था thi=थी thay=थे
raha=रहा rahi=रही rahe=रहे rahega=रहेगा rahegi=रहेगी rahenge=रहेंगे rehna=रहना
aur=और ya=या lekin=लेकिन magar=मगर kyunki=क्योंकि isliye=इसलिए agar=अगर toh=तो phir=फिर fir=फिर bhi=भी
sirf=सिर्फ़ bas=बस abhi=अभी ab=अब jab=जब tab=तब kab=कब kahan=कहाँ yahan=यहाँ yaha=यहाँ wahan=वहाँ waha=वहाँ
idhar=इधर udhar=उधर kaun=कौन kya=क्या kyaa=क्या kaise=कैसे kaisa=कैसा kaisi=कैसी kyun=क्यों kyu=क्यों kyon=क्यों
kitna=कितना kitni=कितनी kitne=कितने kuch=कुछ koi=कोई sab=सब sabhi=सभी har=हर bahut=बहुत bohot=बहुत bahot=बहुत bhot=बहुत
zyada=ज़्यादा jyada=ज़्यादा thoda=थोड़ा thodi=थोड़ी thode=थोड़े nahi=नहीं nahin=नहीं nhi=नहीं mat=मत na=ना
haan=हाँ han=हाँ ji=जी yeh=यह ye=ये yah=यह woh=वह wo=वो vo=वो voh=वह
isko=इसको usko=उसको unko=उनको inko=इनको mujhe=मुझे mujhko=मुझको tumhe=तुम्हें tumko=तुमको hume=हमें humein=हमें
aapko=आपको unhe=उन्हें inhe=इन्हें se=से ko=को ka=का ki=की ke=के mein=में pe=पे tak=तक liye=लिए lie=लिए
saath=साथ sath=साथ baad=बाद pehle=पहले pahle=पहले baare=बारे andar=अंदर bahar=बाहर upar=ऊपर neeche=नीचे niche=नीचे paas=पास
door=दूर aaj=आज kal=कल parson=परसों subah=सुबह shaam=शाम raat=रात din=दिन hafte=हफ़्ते mahina=महीना saal=साल baar=बार
waqt=वक़्त samay=समय log=लोग logon=लोगों aadmi=आदमी ladka=लड़का ladki=लड़की dost=दोस्त ghar=घर kaam=काम baat=बात
cheez=चीज़ kahani=कहानी paisa=पैसा paise=पैसे zindagi=ज़िंदगी pyaar=प्यार pyar=प्यार dil=दिल desh=देश duniya=दुनिया
sawal=सवाल jawab=जवाब madad=मदद zaroor=ज़रूर zaroori=ज़रूरी mushkil=मुश्किल aasan=आसान
accha=अच्छा achha=अच्छा acha=अच्छा achchha=अच्छा acchi=अच्छी achhi=अच्छी bura=बुरा theek=ठीक thik=ठीक sahi=सही galat=गलत
naya=नया nayi=नई naye=नए purana=पुराना bada=बड़ा badi=बड़ी bade=बड़े chhota=छोटा chhoti=छोटी chhote=छोटे
sundar=सुंदर khush=खुश udaas=उदास namaste=नमस्ते namaskar=नमस्कार dhanyavaad=धन्यवाद shukriya=शुक्रिया maaf=माफ़
kripya=कृपया swagat=स्वागत
karna=करना kar=कर karo=करो kare=करे karein=करें karte=करते karta=करता karti=करती karenge=करेंगे karunga=करूँगा
karungi=करूँगी karega=करेगा karegi=करेगी kiya=किया kiye=किए karke=करके karni=करनी
jana=जाना jao=जाओ jaa=जा ja=जा jaata=जाता jaati=जाती jaate=जाते gaya=गया gayi=गई gaye=गए jayenge=जाएँगे jaunga=जाऊँगा
aana=आना aao=आओ aa=आ aata=आता aati=आती aate=आते aaya=आया aayi=आई aaye=आए
dena=देना de=दे diya=दिया diye=दिए dijiye=दीजिए lena=लेना le=ले liya=लिया lo=लो lijiye=लीजिए
bolna=बोलना bolo=बोलो bola=बोला bataya=बताया batao=बताओ batana=बताना dekho=देखो dekha=देखा dekhna=देखना
sunna=सुनना suno=सुनो suna=सुना padhna=पढ़ना likhna=लिखना samajh=समझ samjha=समझा samajhna=समझना
chahiye=चाहिए chahta=चाहता chahti=चाहती sakta=सकता sakti=सकती sakte=सकते milna=मिलना mila=मिला mili=मिली
lagta=लगता lagti=लगती lagega=लगेगा padta=पड़ता padega=पड़ेगा rakhna=रखना rakho=रखो rakha=रखा
chalo=चलो chalna=चलना chalta=चलता chal=चल ruko=रुको ruk=रुक bhejna=भेजना bhejo=भेजो bhej=भेज
khana=खाना peena=पीना sona=सोना uthna=उठना baithna=बैठना hota=होता hoti=होती hote=होते hua=हुआ hui=हुई hue=हुए
hona=होना sochna=सोचना socho=सोचो soch=सोच pata=पता maloom=मालूम jaanta=जानता jaante=जानते
ek=एक teen=तीन paanch=पाँच chhe=छह saat=सात aath=आठ nau=नौ das=दस sau=सौ hazaar=हज़ार lakh=लाख
""")
HI.update(_pairs("""
tum=तुम aap=आप tu=तू mai=मैं kharab=खराब bata=बता batayein=बताएँ bhai=भाई yaar=यार sun=सुन dekh=देख bol=बोल kaafi=काफ़ी kabhi=कभी hamesha=हमेशा
wapas=वापस shuru=शुरू khatam=खत्म pura=पूरा poora=पूरा puri=पूरी pure=पूरे sabse=सबसे bilkul=बिलकुल matlab=मतलब
tarah=तरह tarika=तरीका wajah=वजह fayda=फ़ायदा ghanta=घंटा ghante=घंटे
"""))
HI.update({"dono": "दोनों", "wala": "वाला", "wali": "वाली", "wale": "वाले", "vala": "वाला", "vali": "वाली", "vale": "वाले"})

# Ambiguous roman words: Hindi reading vs English reading. Decided by the rest of the sentence.
AMBIG = {"main": "मैं", "me": "में", "to": "तो", "is": "इस", "the": "थे", "hi": "ही", "he": "है",
         "do": "दो", "us": "उस", "par": "पर", "hum": "हम"}

ENG_COMMON = set("""a an the i you he she it we they me my your his her its our their this that these those
is are was were be been am do does did have has had will would can could should may might must shall not no yes
and or but if so because than then when where what which who whom why how all any some more most much many
one two three four five six seven eight nine ten first second last next new old good bad big small long short
hello hi hey ok okay please sorry thanks thank welcome name time day week month year today tomorrow yesterday
in on at by for with from to of about into over under after before between out up down off again here there
go going went come came get got make made take took see saw say said tell told know knew think thought want need
like love work life people thing way man woman friend home world country family business market company team
project meeting office phone mobile computer laptop internet video audio podcast channel email message call
ready slides slide problem solution question answer example basically actually really very just also still even
only more well now always never sometimes maybe pretty super cool great nice best better free live online
""".split())

# Devanagari spellings of common English words (Whisper writes these) -> Latin
DEV_EN = _pairs("""
हेलो=hello हैलो=hello हाय=hi मीटिंग=meeting प्रेजेंटेशन=presentation स्लाइड्स=slides स्लाइड=slide ऑफिस=office
ऑफ़िस=office फोन=phone फ़ोन=phone मोबाइल=mobile कंप्यूटर=computer लैपटॉप=laptop इंटरनेट=internet वीडियो=video
ऑडियो=audio पॉडकास्ट=podcast चैनल=channel सब्सक्राइब=subscribe लाइक=like शेयर=share कमेंट=comment ईमेल=email
मैसेज=message कॉल=call टाइम=time प्रॉब्लम=problem सॉल्यूशन=solution बेसिकली=basically एक्चुअली=actually
प्लीज़=please प्लीज=please सॉरी=sorry थैंक्स=thanks ओके=okay टेक्नोलॉजी=technology सॉफ्टवेयर=software ऐप=app
एप्लीकेशन=application वेबसाइट=website डेटा=data सर्वर=server बिज़नेस=business बिजनेस=business मैनेजर=manager
टीम=team प्रोजेक्ट=project क्लाइंट=client कस्टमर=customer सर्विस=service क्वालिटी=quality सिस्टम=system
मार्केट=market कंपनी=company करियर=career स्टूडेंट=student कॉलेज=college स्कूल=school एग्जाम=exam रिजल्ट=result
इंटरव्यू=interview जॉब=job सैलरी=salary गेम=game मूवी=movie फिल्म=film म्यूजिक=music सॉन्ग=song लव=love
लाइफ=life सुपर=super फेमस=famous इम्पोर्टेंट=important इंपोर्टेंट=important एक्साइटेड=excited बोरिंग=boring
कूल=cool ग्रेट=great गुड=good बैड=bad सिंपल=simple एक्सपीरियंस=experience फीलिंग=feeling स्टोरी=story टॉपिक=topic
इंस्टाग्राम=Instagram यूट्यूब=YouTube गूगल=Google व्हाट्सएप=WhatsApp फेसबुक=Facebook
""")

# ------------------- rule transliteration (fallback for Hindi-looking OOV) ----------
_CONS = [("chh", "छ"), ("kh", "ख"), ("gh", "घ"), ("ch", "च"), ("jh", "झ"), ("th", "थ"), ("dh", "ध"), ("ph", "फ"),
         ("bh", "भ"), ("sh", "श"), ("k", "क"), ("g", "ग"), ("c", "क"), ("j", "ज"), ("t", "त"), ("d", "द"),
         ("n", "न"), ("p", "प"), ("b", "ब"), ("m", "म"), ("y", "य"), ("r", "र"), ("l", "ल"), ("v", "व"),
         ("w", "व"), ("s", "स"), ("h", "ह"), ("f", "फ़"), ("z", "ज़"), ("q", "क़")]
_VOW = [("aa", ("आ", "ा")), ("ai", ("ऐ", "ै")), ("au", ("औ", "ौ")), ("ee", ("ई", "ी")), ("ii", ("ई", "ी")),
        ("oo", ("ऊ", "ू")), ("uu", ("ऊ", "ू")), ("a", ("अ", "")), ("i", ("इ", "ि")), ("u", ("उ", "ु")),
        ("e", ("ए", "े")), ("o", ("ओ", "ो"))]
_STOPS = {"k", "kh", "g", "gh", "ch", "chh", "j", "jh", "t", "th", "d", "dh", "p", "ph", "b", "bh", "s", "sh"}
_HAL_A = {"k", "kh", "g", "gh", "ch", "j", "t", "th", "d", "dh", "p", "ph", "b", "bh", "s", "sh", "f", "z"}
_HAL_B = {"y", "r", "v", "w"}


def translit(word):
    w, i, toks = word.lower(), 0, []
    while i < len(w):
        for r, d in _CONS:
            if w.startswith(r, i):
                toks.append(("C", r, d)); i += len(r); break
        else:
            for r, d in _VOW:
                if w.startswith(r, i):
                    toks.append(("V", r, d)); i += len(r); break
            else:
                i += 1
    out, pend, pend_r = [], False, ""
    for k, (t, r, d) in enumerate(toks):
        nxt = toks[k + 1] if k + 1 < len(toks) else None
        if t == "C":
            if r in ("n", "m") and k > 0 and nxt and nxt[0] == "C" and nxt[1] in _STOPS:
                out.append("ं"); pend = False; continue          # nasal before a stop -> anusvara
            if pend and ((pend_r in _HAL_A and r in _HAL_B) or (pend_r in ("s", "sh") and r in ("t", "th", "k", "p", "n", "m", "l"))):
                out.append("्")
            out.append(d); pend, pend_r = True, r
        else:
            last = k == len(toks) - 1
            if pend:
                out.append("ी" if (last and r == "i") else d[1] if not (last and r == "a") else "ा")
            else:
                out.append("ई" if (last and r == "i") else d[0])
            pend = False
    return "".join(out)


# ---------------------- Hindi-likeness of an unknown Latin word --------------------
def hindi_score(w):
    s = 0
    for pat, pts in (("chh", 2), ("aa", 2), ("jh", 2), ("kh", 1), ("gh", 1), ("bh", 1), ("dh", 1), ("oo", 0)):
        if pat in w: s += pts
    for suf, pts in (("wala", 3), ("wale", 3), ("wali", 3), ("iye", 2), ("enge", 3), ("egi", 2), ("ega", 2),
                     ("aana", 2), ("ana", 1), ("na", 1), ("oon", 2), ("ein", 1)):
        if w.endswith(suf): s += pts; break
    for suf, pts in (("tion", 3), ("sion", 3), ("ing", 3), ("ly", 2), ("ment", 2), ("ness", 3), ("able", 3),
                     ("ous", 2), ("ck", 2), ("ight", 3), ("ough", 3), ("ize", 3), ("ise", 2)):
        if w.endswith(suf) or (suf in ("ck", "ight", "ough") and suf in w): s -= pts
    if "th" in w: s -= 1
    if "ph" in w: s -= 1
    if any(x in w for x in ("x", "qu", "wh")): s -= 2
    return s


TOKEN = re.compile(r"[A-Za-z][A-Za-z']*|[\u0900-\u097F]+|\d+|\s+|.", re.S)


def convert(text, oov=None):
    oov = oov if oov is not None else Counter()
    toks = TOKEN.findall(text)
    n = len(toks)
    kind = [None] * n            # "hi" | "en" | "name" | "amb" | "keep"
    out = list(toks)
    sent_start = True
    for i, t in enumerate(toks):
        if re.match(r"[\u0900-\u097F]", t):
            if DEV_LOAN_TO_LATIN and t in DEV_EN:
                out[i] = DEV_EN[t]; kind[i] = "en"
            else:
                kind[i] = "hi"
            sent_start = False
        elif re.match(r"[A-Za-z]", t):
            lo = t.lower().replace("'", "")
            if lo in EXTRA_ENGLISH:                 kind[i] = "en"
            elif lo in EXTRA_HINDI:                 out[i], kind[i] = EXTRA_HINDI[lo], "hi"
            elif lo in AMBIG:                       kind[i] = "amb"
            elif lo in HI:                          out[i], kind[i] = HI[lo], "hi"
            elif lo in ENG_COMMON:                  kind[i] = "en"
            elif t[0].isupper() and not sent_start: kind[i] = "name"       # Milan, Delhi ...
            else:
                sc = hindi_score(lo)
                if sc >= 1:
                    out[i], kind[i] = translit(lo), "hi"; oov[(lo, "hindi->" + out[i])] += 1
                else:
                    kind[i] = "en"; oov[(lo, "kept-english")] += 1
            sent_start = False
        elif t in ".!?।\n":
            sent_start = True
    # clause-level context for ambiguous words (clause = text between , . ! ? ; |)
    clause, c = [0] * n, 0
    for i, t in enumerate(toks):
        clause[i] = c
        if t in ",.!?;।\n":
            c += 1
    for i, t in enumerate(toks):
        if kind[i] == "amb":
            hi_n = sum(kind[j] == "hi" for j in range(n) if clause[j] == clause[i])
            en_n = sum(kind[j] == "en" for j in range(n) if clause[j] == clause[i])
            lo = t.lower()
            first = all(k is None for k in kind[:i]) and not any(re.match(r"[A-Za-z\u0900-\u097F]", x) for x in toks[:i])
            if lo == "hi" and first:
                continue                                        # greeting "Hi"
            if hi_n > en_n:
                out[i] = AMBIG[lo]
    return "".join(out)


def for_training(text):
    """Text as the model will see it: Latin lowercased, spaces tidy."""
    t = re.sub(r"[A-Za-z]+", lambda m: m.group(0).lower(), text)
    return re.sub(r"\s+", " ", t).strip()


def run_transcripts():
    base = Path(BASE)
    src = base / "transcripts.jsonl"
    recs = [json.loads(l) for l in src.read_text(encoding="utf-8").splitlines() if l.strip()]
    oov = Counter()
    for r in recs:
        r["text_norm"] = convert(r["text_raw"], oov)
        r["text_train"] = for_training(r["text_norm"])
    (base / "transcripts_norm.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in recs), encoding="utf-8")
    with open(base / "oov_report.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["word", "decision", "count"])
        for (word, dec), c in oov.most_common():
            w.writerow([word, dec, c])
    print(f"{len(recs)} transcripts normalized. Unknown words: {len(oov)} (see oov_report.csv - fix the frequent ones)")


if RUN_ON_TRANSCRIPTS:
    run_transcripts()
else:
    demo = ["Hello mera naam Milan hai aur isko basically...",
            "Hello my name is milan",
            "Namaste mera naam milan hai",
            "main office ja raha hu",
            "Kal meeting hai, please slides ready rakhna.",
            "Is project ka aavashyak sanshodhan kal tak karna hai",
            "हेलो मेरा नाम मिलन है और आज की मीटिंग बहुत ज़रूरी है"]
    for d in demo:
        print(d, "\n   ->", convert(d))

Hello mera naam Milan hai aur isko basically... 
   -> Hello मेरा नाम Milan है और इसको basically...
Hello my name is milan 
   -> Hello my name is milan
Namaste mera naam milan hai 
   -> नमस्ते मेरा नाम milan है
main office ja raha hu 
   -> मैं office जा रहा हूँ
Kal meeting hai, please slides ready rakhna. 
   -> कल meeting है, please slides ready रखना.
Is project ka aavashyak sanshodhan kal tak karna hai 
   -> इस project का आवश्यक संशोधन कल तक करना है
हेलो मेरा नाम मिलन है और आज की मीटिंग बहुत ज़रूरी है 
   -> hello मेरा नाम मिलन है और आज की meeting बहुत ज़रूरी है


In [6]:
# =====================================================================
# RESONA - TRANSCRIPTION CELL  (Whisper large-v3 via faster-whisper)   v1.1
# Every chunk in speakers/<voice>/chunks -> text  (Hindi = Devanagari, English words = Latin)
# Output: BASE/transcripts.jsonl   (resume-safe; re-run continues where it stopped)
#
# 1st run: TEST_N = 30  -> transcribes 30 random clips, saves transcripts_test.jsonl. READ them.
# Then:    TEST_N = 0   -> full run (~4300 clips, roughly 1-2 h on a T4; use Save & Run All).
# Needs: Internet ON (model download), Accelerator GPU T4.
# Run this in its OWN notebook/session, not in the training notebook (GPU memory).
# v1.1: audio is loaded by us and passed as an array (fixes the 'metadata_errors' PyAV error).
# =====================================================================

# ------------------------------- CONFIG ------------------------------
BASE      = "/kaggle/working/resona_data"
MODEL     = "large-v3"
LANGUAGE  = "hi"     # "hi" (best for Hinglish podcasts) | "en" | None = auto-detect per clip
TEST_N    = 30       # >0 = test on N random clips; 0 = full run
BEAM      = 5
# Style hint for Whisper: Hindi words in Devanagari, English words in Latin letters
PROMPT    = "नमस्ते, मेरा नाम मिलन है। Hello everyone, आज हम AI के बारे में बात करेंगे।"
# ---------------------------------------------------------------------

import sys, subprocess, importlib, json, random, time, re, traceback
from collections import Counter
from pathlib import Path


def _need(mod, pip_name=None):
    try:
        return importlib.import_module(mod)
    except ImportError:
        print(f"installing {pip_name or mod} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or mod], check=False)
        importlib.invalidate_caches()
        return importlib.import_module(mod)


import numpy as np
import soundfile as sf
from scipy import signal

DEV_RE, LAT_RE = re.compile(r"[\u0900-\u097F]"), re.compile(r"[A-Za-z]")


def load_16k(path):
    """Read the wav ourselves and give Whisper a 16 kHz float array (avoids faster-whisper's decoder / PyAV)."""
    x, sr = sf.read(str(path), dtype="float32")
    x = x.mean(axis=1) if x.ndim > 1 else x
    if sr != 16000:
        g = int(np.gcd(sr, 16000))
        x = signal.resample_poly(x, 16000 // g, sr // g).astype(np.float32)
    return np.ascontiguousarray(x), len(x) / 16000


def script_mix(t):
    d, l = len(DEV_RE.findall(t)), len(LAT_RE.findall(t))
    if d + l == 0:
        return "none"
    r = d / (d + l)
    return "hindi" if r > 0.9 else ("english" if r < 0.1 else "mixed")


def flags_for(text, dur, avg_lp, comp, nsp):
    f = []
    if not text.strip():                       f.append("empty")
    if avg_lp is not None and avg_lp < -1.0:   f.append("low_confidence")
    if comp is not None and comp > 2.4:        f.append("repetition")
    if nsp is not None and nsp > 0.6:          f.append("maybe_no_speech")
    cps = len(text) / max(dur, 0.1)
    if cps > 28:                               f.append("too_many_chars")
    if cps < 2.0 and text.strip():             f.append("too_few_chars")
    return f


def main():
    base = Path(BASE)
    files = sorted((base / "speakers").glob("*/chunks/*.wav"))
    if not files:
        raise SystemExit(f"No chunks found under {base}/speakers/<voice>/chunks")
    if TEST_N:
        files = random.Random(1).sample(files, min(TEST_N, len(files)))
    out_f = base / ("transcripts_test.jsonl" if TEST_N else "transcripts.jsonl")
    done = set()
    if out_f.exists():
        done = {json.loads(l)["chunk_id"] for l in out_f.read_text(encoding="utf-8").splitlines() if l.strip()}
    todo = [f for f in files if f.stem not in done]
    print(f"{len(files)} clips | {len(done)} already done | {len(todo)} to do | model={MODEL} lang={LANGUAGE}")

    fw = _need("faster_whisper", "faster-whisper")
    try:
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        dev = "cpu"
    print("device:", dev, "(GPU strongly recommended)" if dev == "cpu" else "")
    model = fw.WhisperModel(MODEL, device=dev, compute_type="float16" if dev == "cuda" else "int8")

    t0, fails = time.time(), 0
    with open(out_f, "a", encoding="utf-8") as fo:
        for i, p in enumerate(todo, 1):
            try:
                audio, dur = load_16k(p)
                segs, info = model.transcribe(
                    audio, language=LANGUAGE, beam_size=BEAM, initial_prompt=PROMPT,
                    condition_on_previous_text=False, vad_filter=False, temperature=0.0)
                segs = list(segs)
                text = re.sub(r"\s+", " ", " ".join(s.text.strip() for s in segs)).strip()
                avg_lp = float(sum(s.avg_logprob for s in segs) / len(segs)) if segs else None
                comp = float(max(s.compression_ratio for s in segs)) if segs else None
                nsp = float(max(s.no_speech_prob for s in segs)) if segs else None
                fl = flags_for(text, dur, avg_lp, comp, nsp)
                rec = {"chunk_id": p.stem, "speaker": p.parent.parent.name, "path": str(p),
                       "duration": round(dur, 2), "text_raw": text, "script": script_mix(text),
                       "lang": getattr(info, "language", LANGUAGE),
                       "avg_logprob": None if avg_lp is None else round(avg_lp, 3),
                       "flags": fl}
                fo.write(json.dumps(rec, ensure_ascii=False) + "\n")
                fails = 0
            except Exception as ex:
                print(f"   [skipped] {p.name}: {str(ex)[:120]}")
                fails += 1
                if fails >= 5:
                    raise SystemExit("5 clips failed in a row - stopping. Send me the error above.")
            if i % 50 == 0:
                fo.flush()
                el = time.time() - t0
                print(f"[{i}/{len(todo)}] {el/60:.1f} min elapsed, ~{el/i*(len(todo)-i)/60:.0f} min left")

    recs = [json.loads(l) for l in out_f.read_text(encoding="utf-8").splitlines() if l.strip()]
    print("\n================ SUMMARY ================")
    print("clips transcribed:", len(recs), "| flagged:", sum(bool(r["flags"]) for r in recs))
    print("script mix:", dict(Counter(r["script"] for r in recs)))
    print("flags:", dict(Counter(f for r in recs for f in r["flags"])))
    print("per voice:", dict(Counter(r["speaker"] for r in recs)))
    print("\nSample transcripts (READ THESE):")
    for r in random.Random(2).sample(recs, min(12, len(recs))):
        print(f"- [{r['speaker']}, {r['script']}, {r['flags'] or 'ok'}] {r['text_raw']}")
    if TEST_N:
        print("\nThis was a TEST. Check the text above: Hindi in Devanagari, English words in Latin, no wrong words.")
        print("If OK set TEST_N = 0 for the full run. If not, tell me what looks wrong.")


try:
    main()
except SystemExit as e:
    print(e)
except Exception:
    print("UNEXPECTED ERROR (send this to me):")
    traceback.print_exc()

30 clips | 0 already done | 30 to do | model=large-v3 lang=hi
installing faster-whisper ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 84.4 MB/s eta 0:00:00
device: cuda 

================ SUMMARY ================
clips transcribed: 30 | flagged: 2
script mix: {'hindi': 7, 'english': 22, 'mixed': 1}
flags: {'repetition': 2}
per voice: {'kabir': 1, 'arjun': 11, 'neha': 6, 'vivaan': 4, 'rohan': 7, 'riya': 1}

Sample transcripts (READ THESE):
- [arjun, english, ok] You know, typically the guys who are innovators tend to think of it like that. Let me make things smaller.
- [neha, english, ok] And then you started binge watching for 45 minutes in the evening instead of working on the thing that's overwhelming you. So then you did the binge watchi

In [7]:
# =====================================================================
# RESONA TEXT FRONTEND v1  -  Hinglish -> natural mixed script  (ONE CELL)
#   Hindi words   -> Devanagari      (mera -> मेरा, hai -> है)
#   English words -> stay in English (meeting, basically, office)
#   Names         -> stay as written (Milan)
# Same rule for Roman input, Devanagari input (Whisper output) and mixed input.
#
#   convert("Hello mera naam Milan hai aur isko basically...")
#     -> "Hello मेरा नाम Milan है और इसको basically..."
#
# Uses: (1) a Hindi-roman word list, (2) context for ambiguous words (main / to / is / me),
#       (3) a spelling-pattern guess + rule transliteration for unknown Hindi-looking words.
# Every unknown word is logged in an OOV report so you can fix it (add to EXTRA_HINDI / EXTRA_ENGLISH).
# =====================================================================

# ------------------------------- CONFIG ------------------------------
RUN_ON_TRANSCRIPTS = False   # True = read BASE/transcripts.jsonl, write transcripts_norm.jsonl + oov_report.csv
BASE               = "/kaggle/working/resona_data"
DEV_LOAN_TO_LATIN  = True    # Devanagari loanwords from Whisper (मीटिंग) -> Latin (meeting)
EXTRA_HINDI   = {}           # force: {"sarkar": "सरकार"}
EXTRA_ENGLISH = set()        # force keep-as-English: {"milan", "kotlin"}
# ---------------------------------------------------------------------

import re, json, csv
from collections import Counter
from pathlib import Path


def _pairs(s):
    return dict(p.split("=", 1) for p in s.split())


# ------------------------- HINDI (roman -> Devanagari) ---------------
HI = _pairs("""
mera=मेरा meri=मेरी mere=मेरे tera=तेरा teri=तेरी tere=तेरे tumhara=तुम्हारा tumhari=तुम्हारी tumhare=तुम्हारे
hamara=हमारा hamari=हमारी hamare=हमारे aapka=आपका aapki=आपकी aapke=आपके apna=अपना apni=अपनी apne=अपने
uska=उसका uski=उसकी uske=उसके iska=इसका iski=इसकी iske=इसके unka=उनका unki=उनकी unke=उनके
naam=नाम hai=है hain=हैं hu=हूँ hun=हूँ hoon=हूँ ho=हो hoga=होगा hogi=होगी honge=होंगे tha=था thi=थी thay=थे
raha=रहा rahi=रही rahe=रहे rahega=रहेगा rahegi=रहेगी rahenge=रहेंगे rehna=रहना
aur=और ya=या lekin=लेकिन magar=मगर kyunki=क्योंकि isliye=इसलिए agar=अगर toh=तो phir=फिर fir=फिर bhi=भी
sirf=सिर्फ़ bas=बस abhi=अभी ab=अब jab=जब tab=तब kab=कब kahan=कहाँ yahan=यहाँ yaha=यहाँ wahan=वहाँ waha=वहाँ
idhar=इधर udhar=उधर kaun=कौन kya=क्या kyaa=क्या kaise=कैसे kaisa=कैसा kaisi=कैसी kyun=क्यों kyu=क्यों kyon=क्यों
kitna=कितना kitni=कितनी kitne=कितने kuch=कुछ koi=कोई sab=सब sabhi=सभी har=हर bahut=बहुत bohot=बहुत bahot=बहुत bhot=बहुत
zyada=ज़्यादा jyada=ज़्यादा thoda=थोड़ा thodi=थोड़ी thode=थोड़े nahi=नहीं nahin=नहीं nhi=नहीं mat=मत na=ना
haan=हाँ han=हाँ ji=जी yeh=यह ye=ये yah=यह woh=वह wo=वो vo=वो voh=वह
isko=इसको usko=उसको unko=उनको inko=इनको mujhe=मुझे mujhko=मुझको tumhe=तुम्हें tumko=तुमको hume=हमें humein=हमें
aapko=आपको unhe=उन्हें inhe=इन्हें se=से ko=को ka=का ki=की ke=के mein=में pe=पे tak=तक liye=लिए lie=लिए
saath=साथ sath=साथ baad=बाद pehle=पहले pahle=पहले baare=बारे andar=अंदर bahar=बाहर upar=ऊपर neeche=नीचे niche=नीचे paas=पास
door=दूर aaj=आज kal=कल parson=परसों subah=सुबह shaam=शाम raat=रात din=दिन hafte=हफ़्ते mahina=महीना saal=साल baar=बार
waqt=वक़्त samay=समय log=लोग logon=लोगों aadmi=आदमी ladka=लड़का ladki=लड़की dost=दोस्त ghar=घर kaam=काम baat=बात
cheez=चीज़ kahani=कहानी paisa=पैसा paise=पैसे zindagi=ज़िंदगी pyaar=प्यार pyar=प्यार dil=दिल desh=देश duniya=दुनिया
sawal=सवाल jawab=जवाब madad=मदद zaroor=ज़रूर zaroori=ज़रूरी mushkil=मुश्किल aasan=आसान
accha=अच्छा achha=अच्छा acha=अच्छा achchha=अच्छा acchi=अच्छी achhi=अच्छी bura=बुरा theek=ठीक thik=ठीक sahi=सही galat=गलत
naya=नया nayi=नई naye=नए purana=पुराना bada=बड़ा badi=बड़ी bade=बड़े chhota=छोटा chhoti=छोटी chhote=छोटे
sundar=सुंदर khush=खुश udaas=उदास namaste=नमस्ते namaskar=नमस्कार dhanyavaad=धन्यवाद shukriya=शुक्रिया maaf=माफ़
kripya=कृपया swagat=स्वागत
karna=करना kar=कर karo=करो kare=करे karein=करें karte=करते karta=करता karti=करती karenge=करेंगे karunga=करूँगा
karungi=करूँगी karega=करेगा karegi=करेगी kiya=किया kiye=किए karke=करके karni=करनी
jana=जाना jao=जाओ jaa=जा ja=जा jaata=जाता jaati=जाती jaate=जाते gaya=गया gayi=गई gaye=गए jayenge=जाएँगे jaunga=जाऊँगा
aana=आना aao=आओ aa=आ aata=आता aati=आती aate=आते aaya=आया aayi=आई aaye=आए
dena=देना de=दे diya=दिया diye=दिए dijiye=दीजिए lena=लेना le=ले liya=लिया lo=लो lijiye=लीजिए
bolna=बोलना bolo=बोलो bola=बोला bataya=बताया batao=बताओ batana=बताना dekho=देखो dekha=देखा dekhna=देखना
sunna=सुनना suno=सुनो suna=सुना padhna=पढ़ना likhna=लिखना samajh=समझ samjha=समझा samajhna=समझना
chahiye=चाहिए chahta=चाहता chahti=चाहती sakta=सकता sakti=सकती sakte=सकते milna=मिलना mila=मिला mili=मिली
lagta=लगता lagti=लगती lagega=लगेगा padta=पड़ता padega=पड़ेगा rakhna=रखना rakho=रखो rakha=रखा
chalo=चलो chalna=चलना chalta=चलता chal=चल ruko=रुको ruk=रुक bhejna=भेजना bhejo=भेजो bhej=भेज
khana=खाना peena=पीना sona=सोना uthna=उठना baithna=बैठना hota=होता hoti=होती hote=होते hua=हुआ hui=हुई hue=हुए
hona=होना sochna=सोचना socho=सोचो soch=सोच pata=पता maloom=मालूम jaanta=जानता jaante=जानते
ek=एक teen=तीन paanch=पाँच chhe=छह saat=सात aath=आठ nau=नौ das=दस sau=सौ hazaar=हज़ार lakh=लाख
""")
HI.update(_pairs("""
tum=तुम aap=आप tu=तू mai=मैं kharab=खराब bata=बता batayein=बताएँ bhai=भाई yaar=यार sun=सुन dekh=देख bol=बोल kaafi=काफ़ी kabhi=कभी hamesha=हमेशा
wapas=वापस shuru=शुरू khatam=खत्म pura=पूरा poora=पूरा puri=पूरी pure=पूरे sabse=सबसे bilkul=बिलकुल matlab=मतलब
tarah=तरह tarika=तरीका wajah=वजह fayda=फ़ायदा ghanta=घंटा ghante=घंटे
"""))
HI.update({"dono": "दोनों", "wala": "वाला", "wali": "वाली", "wale": "वाले", "vala": "वाला", "vali": "वाली", "vale": "वाले"})

# Ambiguous roman words: Hindi reading vs English reading. Decided by the rest of the sentence.
AMBIG = {"main": "मैं", "me": "में", "to": "तो", "is": "इस", "the": "थे", "hi": "ही", "he": "है",
         "do": "दो", "us": "उस", "par": "पर", "hum": "हम"}

ENG_COMMON = set("""a an the i you he she it we they me my your his her its our their this that these those
is are was were be been am do does did have has had will would can could should may might must shall not no yes
and or but if so because than then when where what which who whom why how all any some more most much many
one two three four five six seven eight nine ten first second last next new old good bad big small long short
hello hi hey ok okay please sorry thanks thank welcome name time day week month year today tomorrow yesterday
in on at by for with from to of about into over under after before between out up down off again here there
go going went come came get got make made take took see saw say said tell told know knew think thought want need
like love work life people thing way man woman friend home world country family business market company team
project meeting office phone mobile computer laptop internet video audio podcast channel email message call
ready slides slide problem solution question answer example basically actually really very just also still even
only more well now always never sometimes maybe pretty super cool great nice best better free live online
""".split())

# Devanagari spellings of common English words (Whisper writes these) -> Latin
DEV_EN = _pairs("""
हेलो=hello हैलो=hello हाय=hi मीटिंग=meeting प्रेजेंटेशन=presentation स्लाइड्स=slides स्लाइड=slide ऑफिस=office
ऑफ़िस=office फोन=phone फ़ोन=phone मोबाइल=mobile कंप्यूटर=computer लैपटॉप=laptop इंटरनेट=internet वीडियो=video
ऑडियो=audio पॉडकास्ट=podcast चैनल=channel सब्सक्राइब=subscribe लाइक=like शेयर=share कमेंट=comment ईमेल=email
मैसेज=message कॉल=call टाइम=time प्रॉब्लम=problem सॉल्यूशन=solution बेसिकली=basically एक्चुअली=actually
प्लीज़=please प्लीज=please सॉरी=sorry थैंक्स=thanks ओके=okay टेक्नोलॉजी=technology सॉफ्टवेयर=software ऐप=app
एप्लीकेशन=application वेबसाइट=website डेटा=data सर्वर=server बिज़नेस=business बिजनेस=business मैनेजर=manager
टीम=team प्रोजेक्ट=project क्लाइंट=client कस्टमर=customer सर्विस=service क्वालिटी=quality सिस्टम=system
मार्केट=market कंपनी=company करियर=career स्टूडेंट=student कॉलेज=college स्कूल=school एग्जाम=exam रिजल्ट=result
इंटरव्यू=interview जॉब=job सैलरी=salary गेम=game मूवी=movie फिल्म=film म्यूजिक=music सॉन्ग=song लव=love
लाइफ=life सुपर=super फेमस=famous इम्पोर्टेंट=important इंपोर्टेंट=important एक्साइटेड=excited बोरिंग=boring
कूल=cool ग्रेट=great गुड=good बैड=bad सिंपल=simple एक्सपीरियंस=experience फीलिंग=feeling स्टोरी=story टॉपिक=topic
इंस्टाग्राम=Instagram यूट्यूब=YouTube गूगल=Google व्हाट्सएप=WhatsApp फेसबुक=Facebook
""")

# ------------------- rule transliteration (fallback for Hindi-looking OOV) ----------
_CONS = [("chh", "छ"), ("kh", "ख"), ("gh", "घ"), ("ch", "च"), ("jh", "झ"), ("th", "थ"), ("dh", "ध"), ("ph", "फ"),
         ("bh", "भ"), ("sh", "श"), ("k", "क"), ("g", "ग"), ("c", "क"), ("j", "ज"), ("t", "त"), ("d", "द"),
         ("n", "न"), ("p", "प"), ("b", "ब"), ("m", "म"), ("y", "य"), ("r", "र"), ("l", "ल"), ("v", "व"),
         ("w", "व"), ("s", "स"), ("h", "ह"), ("f", "फ़"), ("z", "ज़"), ("q", "क़")]
_VOW = [("aa", ("आ", "ा")), ("ai", ("ऐ", "ै")), ("au", ("औ", "ौ")), ("ee", ("ई", "ी")), ("ii", ("ई", "ी")),
        ("oo", ("ऊ", "ू")), ("uu", ("ऊ", "ू")), ("a", ("अ", "")), ("i", ("इ", "ि")), ("u", ("उ", "ु")),
        ("e", ("ए", "े")), ("o", ("ओ", "ो"))]
_STOPS = {"k", "kh", "g", "gh", "ch", "chh", "j", "jh", "t", "th", "d", "dh", "p", "ph", "b", "bh", "s", "sh"}
_HAL_A = {"k", "kh", "g", "gh", "ch", "j", "t", "th", "d", "dh", "p", "ph", "b", "bh", "s", "sh", "f", "z"}
_HAL_B = {"y", "r", "v", "w"}


def translit(word):
    w, i, toks = word.lower(), 0, []
    while i < len(w):
        for r, d in _CONS:
            if w.startswith(r, i):
                toks.append(("C", r, d)); i += len(r); break
        else:
            for r, d in _VOW:
                if w.startswith(r, i):
                    toks.append(("V", r, d)); i += len(r); break
            else:
                i += 1
    out, pend, pend_r = [], False, ""
    for k, (t, r, d) in enumerate(toks):
        nxt = toks[k + 1] if k + 1 < len(toks) else None
        if t == "C":
            if r in ("n", "m") and k > 0 and nxt and nxt[0] == "C" and nxt[1] in _STOPS:
                out.append("ं"); pend = False; continue          # nasal before a stop -> anusvara
            if pend and ((pend_r in _HAL_A and r in _HAL_B) or (pend_r in ("s", "sh") and r in ("t", "th", "k", "p", "n", "m", "l"))):
                out.append("्")
            out.append(d); pend, pend_r = True, r
        else:
            last = k == len(toks) - 1
            if pend:
                out.append("ी" if (last and r == "i") else d[1] if not (last and r == "a") else "ा")
            else:
                out.append("ई" if (last and r == "i") else d[0])
            pend = False
    return "".join(out)


# ---------------------- Hindi-likeness of an unknown Latin word --------------------
def hindi_score(w):
    s = 0
    for pat, pts in (("chh", 2), ("aa", 2), ("jh", 2), ("kh", 1), ("gh", 1), ("bh", 1), ("dh", 1), ("oo", 0)):
        if pat in w: s += pts
    for suf, pts in (("wala", 3), ("wale", 3), ("wali", 3), ("iye", 2), ("enge", 3), ("egi", 2), ("ega", 2),
                     ("aana", 2), ("ana", 1), ("na", 1), ("oon", 2), ("ein", 1)):
        if w.endswith(suf): s += pts; break
    for suf, pts in (("tion", 3), ("sion", 3), ("ing", 3), ("ly", 2), ("ment", 2), ("ness", 3), ("able", 3),
                     ("ous", 2), ("ck", 2), ("ight", 3), ("ough", 3), ("ize", 3), ("ise", 2)):
        if w.endswith(suf) or (suf in ("ck", "ight", "ough") and suf in w): s -= pts
    if "th" in w: s -= 1
    if "ph" in w: s -= 1
    if any(x in w for x in ("x", "qu", "wh")): s -= 2
    return s


TOKEN = re.compile(r"[A-Za-z][A-Za-z']*|[\u0900-\u097F]+|\d+|\s+|.", re.S)


def convert(text, oov=None, roman_mode=True):
    """roman_mode=True : user/typed Hinglish (Roman Hindi words -> Devanagari).
    roman_mode=False: Whisper transcripts (Latin = real English, never touched; only Devanagari loanwords -> Latin)."""
    oov = oov if oov is not None else Counter()
    toks = TOKEN.findall(text)
    n = len(toks)
    kind = [None] * n            # "hi" | "en" | "name" | "amb" | "keep"
    out = list(toks)
    sent_start = True
    for i, t in enumerate(toks):
        if re.match(r"[\u0900-\u097F]", t):
            if DEV_LOAN_TO_LATIN and t in DEV_EN:
                out[i] = DEV_EN[t]; kind[i] = "en"
            else:
                kind[i] = "hi"
            sent_start = False
        elif re.match(r"[A-Za-z]", t):
            lo = t.lower().replace("'", "")
            if not roman_mode:                      kind[i] = "en"
            elif lo in EXTRA_ENGLISH:                 kind[i] = "en"
            elif lo in EXTRA_HINDI:                 out[i], kind[i] = EXTRA_HINDI[lo], "hi"
            elif lo in AMBIG:                       kind[i] = "amb"
            elif lo in HI:                          out[i], kind[i] = HI[lo], "hi"
            elif lo in ENG_COMMON:                  kind[i] = "en"
            elif t[0].isupper() and not sent_start: kind[i] = "name"       # Milan, Delhi ...
            else:
                sc = hindi_score(lo)
                if sc >= 1:
                    out[i], kind[i] = translit(lo), "hi"; oov[(lo, "hindi->" + out[i])] += 1
                else:
                    kind[i] = "en"; oov[(lo, "kept-english")] += 1
            sent_start = False
        elif t in ".!?।\n":
            sent_start = True
    # clause-level context for ambiguous words (clause = text between , . ! ? ; |)
    clause, c = [0] * n, 0
    for i, t in enumerate(toks):
        clause[i] = c
        if t in ",.!?;।\n":
            c += 1
    for i, t in enumerate(toks):
        if kind[i] == "amb":
            hi_n = sum(kind[j] == "hi" for j in range(n) if clause[j] == clause[i])
            en_n = sum(kind[j] == "en" for j in range(n) if clause[j] == clause[i])
            lo = t.lower()
            first = all(k is None for k in kind[:i]) and not any(re.match(r"[A-Za-z\u0900-\u097F]", x) for x in toks[:i])
            if lo == "hi" and first:
                continue                                        # greeting "Hi"
            if hi_n > en_n:
                out[i] = AMBIG[lo]
    return "".join(out)


def for_training(text):
    """Text as the model will see it: Latin lowercased, spaces tidy."""
    t = re.sub(r"[A-Za-z]+", lambda m: m.group(0).lower(), text)
    return re.sub(r"\s+", " ", t).strip()


def run_transcripts():
    base = Path(BASE)
    src = base / "transcripts.jsonl"
    recs = [json.loads(l) for l in src.read_text(encoding="utf-8").splitlines() if l.strip()]
    oov = Counter()
    for r in recs:
        r["text_norm"] = convert(r["text_raw"], oov, roman_mode=False)   # Whisper output: Latin is already English
        r["text_train"] = for_training(r["text_norm"])
    (base / "transcripts_norm.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in recs), encoding="utf-8")
    with open(base / "oov_report.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["word", "decision", "count"])
        for (word, dec), c in oov.most_common():
            w.writerow([word, dec, c])
    print(f"{len(recs)} transcripts normalized. Unknown words: {len(oov)} (see oov_report.csv - fix the frequent ones)")


if RUN_ON_TRANSCRIPTS:
    run_transcripts()
else:
    demo = ["Hello mera naam Milan hai aur isko basically...",
            "Hello my name is milan",
            "Namaste mera naam milan hai",
            "main office ja raha hu",
            "Kal meeting hai, please slides ready rakhna.",
            "Is project ka aavashyak sanshodhan kal tak karna hai",
            "हेलो मेरा नाम मिलन है और आज की मीटिंग बहुत ज़रूरी है"]
    for d in demo:
        print(d, "\n   ->", convert(d))

Hello mera naam Milan hai aur isko basically... 
   -> Hello मेरा नाम Milan है और इसको basically...
Hello my name is milan 
   -> Hello my name is milan
Namaste mera naam milan hai 
   -> नमस्ते मेरा नाम milan है
main office ja raha hu 
   -> मैं office जा रहा हूँ
Kal meeting hai, please slides ready rakhna. 
   -> कल meeting है, please slides ready रखना.
Is project ka aavashyak sanshodhan kal tak karna hai 
   -> इस project का आवश्यक संशोधन कल तक करना है
हेलो मेरा नाम मिलन है और आज की मीटिंग बहुत ज़रूरी है 
   -> hello मेरा नाम मिलन है और आज की meeting बहुत ज़रूरी है


In [8]:
# =====================================================================
# RESONA - TRANSCRIPTION CELL  (Whisper large-v3 via faster-whisper)   v1.1
# Every chunk in speakers/<voice>/chunks -> text  (Hindi = Devanagari, English words = Latin)
# Output: BASE/transcripts.jsonl   (resume-safe; re-run continues where it stopped)
#
# FULL RUN (TEST_N = 0): all ~4300 clips, roughly 1-2 h on a T4. Use Save Version > Save & Run All.
# (TEST_N = 30 would only test 30 random clips and save transcripts_test.jsonl.)
# Needs: Internet ON (model download), Accelerator GPU T4.
# Run this in its OWN notebook/session, not in the training notebook (GPU memory).
# v1.1: audio is loaded by us and passed as an array (fixes the 'metadata_errors' PyAV error).
# =====================================================================

# ------------------------------- CONFIG ------------------------------
BASE      = "/kaggle/working/resona_data"
MODEL     = "large-v3"
LANGUAGE  = "hi"     # "hi" (best for Hinglish podcasts) | "en" | None = auto-detect per clip
TEST_N    = 0        # >0 = test on N random clips; 0 = FULL run (all clips)
BEAM      = 5
# Style hint for Whisper: Hindi words in Devanagari, English words in Latin letters
PROMPT    = "नमस्ते, मेरा नाम मिलन है। Hello everyone, आज हम AI के बारे में बात करेंगे।"
# ---------------------------------------------------------------------

import sys, subprocess, importlib, json, random, time, re, traceback
from collections import Counter
from pathlib import Path


def _need(mod, pip_name=None):
    try:
        return importlib.import_module(mod)
    except ImportError:
        print(f"installing {pip_name or mod} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or mod], check=False)
        importlib.invalidate_caches()
        return importlib.import_module(mod)


import numpy as np
import soundfile as sf
from scipy import signal

DEV_RE, LAT_RE = re.compile(r"[\u0900-\u097F]"), re.compile(r"[A-Za-z]")


def load_16k(path):
    """Read the wav ourselves and give Whisper a 16 kHz float array (avoids faster-whisper's decoder / PyAV)."""
    x, sr = sf.read(str(path), dtype="float32")
    x = x.mean(axis=1) if x.ndim > 1 else x
    if sr != 16000:
        g = int(np.gcd(sr, 16000))
        x = signal.resample_poly(x, 16000 // g, sr // g).astype(np.float32)
    return np.ascontiguousarray(x), len(x) / 16000


def script_mix(t):
    d, l = len(DEV_RE.findall(t)), len(LAT_RE.findall(t))
    if d + l == 0:
        return "none"
    r = d / (d + l)
    return "hindi" if r > 0.9 else ("english" if r < 0.1 else "mixed")


def flags_for(text, dur, avg_lp, comp, nsp):
    f = []
    if not text.strip():                       f.append("empty")
    if avg_lp is not None and avg_lp < -1.0:   f.append("low_confidence")
    if comp is not None and comp > 2.4:        f.append("repetition")
    if nsp is not None and nsp > 0.6:          f.append("maybe_no_speech")
    cps = len(text) / max(dur, 0.1)
    if cps > 28:                               f.append("too_many_chars")
    if cps < 2.0 and text.strip():             f.append("too_few_chars")
    return f


def main():
    base = Path(BASE)
    files = sorted((base / "speakers").glob("*/chunks/*.wav"))
    if not files:
        raise SystemExit(f"No chunks found under {base}/speakers/<voice>/chunks")
    if TEST_N:
        files = random.Random(1).sample(files, min(TEST_N, len(files)))
    out_f = base / ("transcripts_test.jsonl" if TEST_N else "transcripts.jsonl")
    done = set()
    if out_f.exists():
        done = {json.loads(l)["chunk_id"] for l in out_f.read_text(encoding="utf-8").splitlines() if l.strip()}
    todo = [f for f in files if f.stem not in done]
    print(f"{len(files)} clips | {len(done)} already done | {len(todo)} to do | model={MODEL} lang={LANGUAGE}")

    fw = _need("faster_whisper", "faster-whisper")
    try:
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        dev = "cpu"
    print("device:", dev, "(GPU strongly recommended)" if dev == "cpu" else "")
    model = fw.WhisperModel(MODEL, device=dev, compute_type="float16" if dev == "cuda" else "int8")

    t0, fails = time.time(), 0
    with open(out_f, "a", encoding="utf-8") as fo:
        for i, p in enumerate(todo, 1):
            try:
                audio, dur = load_16k(p)
                segs, info = model.transcribe(
                    audio, language=LANGUAGE, beam_size=BEAM, initial_prompt=PROMPT,
                    condition_on_previous_text=False, vad_filter=False, temperature=0.0)
                segs = list(segs)
                text = re.sub(r"\s+", " ", " ".join(s.text.strip() for s in segs)).strip()
                avg_lp = float(sum(s.avg_logprob for s in segs) / len(segs)) if segs else None
                comp = float(max(s.compression_ratio for s in segs)) if segs else None
                nsp = float(max(s.no_speech_prob for s in segs)) if segs else None
                fl = flags_for(text, dur, avg_lp, comp, nsp)
                rec = {"chunk_id": p.stem, "speaker": p.parent.parent.name, "path": str(p),
                       "duration": round(dur, 2), "text_raw": text, "script": script_mix(text),
                       "lang": getattr(info, "language", LANGUAGE),
                       "avg_logprob": None if avg_lp is None else round(avg_lp, 3),
                       "flags": fl}
                fo.write(json.dumps(rec, ensure_ascii=False) + "\n")
                fails = 0
            except Exception as ex:
                print(f"   [skipped] {p.name}: {str(ex)[:120]}")
                fails += 1
                if fails >= 5:
                    raise SystemExit("5 clips failed in a row - stopping. Send me the error above.")
            if i % 50 == 0:
                fo.flush()
                el = time.time() - t0
                print(f"[{i}/{len(todo)}] {el/60:.1f} min elapsed, ~{el/i*(len(todo)-i)/60:.0f} min left")

    recs = [json.loads(l) for l in out_f.read_text(encoding="utf-8").splitlines() if l.strip()]
    print("\n================ SUMMARY ================")
    print("clips transcribed:", len(recs), "| flagged:", sum(bool(r["flags"]) for r in recs))
    print("script mix:", dict(Counter(r["script"] for r in recs)))
    print("flags:", dict(Counter(f for r in recs for f in r["flags"])))
    print("per voice:", dict(Counter(r["speaker"] for r in recs)))
    try:
        import pandas as pd
        d = pd.DataFrame(recs)
        print("\nClips per voice x script:")
        print(pd.crosstab(d.speaker, d.script))
        print("\nHours per voice x script:")
        print((d.groupby(["speaker", "script"]).duration.sum() / 3600).round(2).unstack(fill_value=0))
    except Exception as ex:
        print("[table skipped]", ex)
    fl = [r for r in recs if r["flags"]]
    print(f"\nFlagged clips: {len(fl)} (first 8):")
    for r in fl[:8]:
        print(f"- {r['chunk_id']} {r['flags']}: {r['text_raw'][:110]}")
    print("\nSample transcripts (READ THESE):")
    for r in random.Random(2).sample(recs, min(12, len(recs))):
        print(f"- [{r['speaker']}, {r['script']}, {r['flags'] or 'ok'}] {r['text_raw']}")
    if TEST_N:
        print("\nThis was a TEST. Check the text above: Hindi in Devanagari, English words in Latin, no wrong words.")
        print("If OK set TEST_N = 0 for the full run. If not, tell me what looks wrong.")


try:
    main()
except SystemExit as e:
    print(e)
except Exception:
    print("UNEXPECTED ERROR (send this to me):")
    traceback.print_exc()

4324 clips | 0 already done | 4324 to do | model=large-v3 lang=hi
device: cuda 
[50/4324] 1.0 min elapsed, ~85 min left
[100/4324] 1.9 min elapsed, ~81 min left
[150/4324] 2.8 min elapsed, ~78 min left
[200/4324] 3.6 min elapsed, ~73 min left
[250/4324] 4.5 min elapsed, ~74 min left
[300/4324] 5.5 min elapsed, ~74 min left
[350/4324] 6.4 min elapsed, ~73 min left
[400/4324] 7.4 min elapsed, ~73 min left
[450/4324] 8.3 min elapsed, ~72 min left
[500/4324] 9.2 min elapsed, ~71 min left
[550/4324] 10.2 min elapsed, ~70 min left
[600/4324] 11.1 min elapsed, ~69 min left
[650/4324] 11.8 min elapsed, ~67 min left
[700/4324] 12.7 min elapsed, ~66 min left
[750/4324] 13.7 min elapsed, ~65 min left
[800/4324] 14.7 min elapsed, ~65 min left
[850/4324] 15.9 min elapsed, ~65 min left
[900/4324] 16.8 min elapsed, ~64 min left
[950/4324] 17.7 min elapsed, ~63 min left
[1000/4324] 18.6 min elapsed, ~62 min left
[1050/4324] 19.5 min elapsed, ~61 min left
[1100/4324] 20.7 min elapsed, ~61 min left
[115

In [9]:
# =====================================================================
# RESONA - TRANSCRIPT QC / PASS 2   (run AFTER the full transcription)
# Fixes what pass 1 got wrong:
#   1. prompt leakage   ("आज हम बात करेंगे।" invented from the Whisper prompt)
#   2. repetition loops ("आज आज आज ...")
#   3. English speech written in Devanagari (language was forced to Hindi)
# Method: leak filter -> detect the real language of each suspicious clip -> re-transcribe with the
#         right language, NO prompt, temperature fallback -> mark every clip ok / review / drop.
# Output: BASE/transcripts_v2.jsonl  (use status == "ok" for training; "review" = listen first)
# Needs GPU T4 + Internet ON. Resume-safe. About 30-45 min for ~2000 clips (estimate).
# =====================================================================

# ------------------------------- CONFIG ------------------------------
BASE        = "/kaggle/working/resona_data"
MODEL       = "large-v3"
PASS1_PROMPT = "नमस्ते, मेरा नाम मिलन है। Hello everyone, आज हम AI के बारे में बात करेंगे।"   # used in pass 1 (for leak check)
REDO_SCRIPTS = ("hindi", "mixed", "none")   # clips whose pass-1 text was mostly Devanagari / mixed get a language check
TEST_N      = 0        # >0 = only redo N clips (quick test)
# ---------------------------------------------------------------------

import sys, subprocess, importlib, json, random, time, re, traceback
from collections import Counter
from pathlib import Path


def _need(mod, pip_name=None):
    try:
        return importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or mod], check=False)
        importlib.invalidate_caches()
        return importlib.import_module(mod)


import numpy as np
import soundfile as sf
from scipy import signal

DEV_RE, LAT_RE = re.compile(r"[\u0900-\u097F]"), re.compile(r"[A-Za-z]")
HI_LIKE = {"hi", "ur", "ne", "mr", "pa", "sa", "bn", "gu"}      # Whisper often calls Hinglish one of these


def words(t):
    return re.findall(r"[\u0900-\u097FA-Za-z0-9']+", t.lower())


PROMPT_WORDS = set(words(PASS1_PROMPT))


def is_prompt_leak(t):
    w = words(t)
    if not w:
        return False
    if "मिलन" in t or "hello everyone" in t.lower():
        return True
    return len(w) <= 12 and sum(x in PROMPT_WORDS for x in w) / len(w) >= 0.7


def has_loop(t):
    w = words(t)
    if len(w) >= 6 and len(set(w)) / len(w) < 0.4:               # very few distinct words
        return True
    for n in (1, 2, 3, 4):                                          # same n-gram 4+ times in a row
        for i in range(len(w) - n * 4 + 1):
            g = w[i:i + n]
            if all(w[i + k * n:i + (k + 1) * n] == g for k in range(4)):
                return True
    return False


def load_16k(path):
    x, sr = sf.read(str(path), dtype="float32")
    x = x.mean(axis=1) if x.ndim > 1 else x
    if sr != 16000:
        g = int(np.gcd(sr, 16000))
        x = signal.resample_poly(x, 16000 // g, sr // g).astype(np.float32)
    return np.ascontiguousarray(x), len(x) / 16000


def script_mix(t):
    d, l = len(DEV_RE.findall(t)), len(LAT_RE.findall(t))
    if d + l == 0:
        return "none"
    r = d / (d + l)
    return "hindi" if r > 0.9 else ("english" if r < 0.1 else "mixed")


def detect(model, audio):
    try:
        out = model.detect_language(audio)
        return out[0], float(out[1])
    except Exception:
        _, info = model.transcribe(audio, language=None, beam_size=1, without_timestamps=True)
        return info.language, float(info.language_probability)


def transcribe_clean(model, audio, lang):
    kw = dict(language=lang, beam_size=5, initial_prompt=None, condition_on_previous_text=False,
              vad_filter=False, temperature=[0.0, 0.2, 0.4], compression_ratio_threshold=2.0,
              log_prob_threshold=-1.0, no_speech_threshold=0.6)
    try:
        segs, info = model.transcribe(audio, repetition_penalty=1.1, **kw)
    except TypeError:                                              # older faster-whisper
        segs, info = model.transcribe(audio, **kw)
    segs = list(segs)
    text = re.sub(r"\s+", " ", " ".join(s.text.strip() for s in segs)).strip()
    lp = float(sum(s.avg_logprob for s in segs) / len(segs)) if segs else None
    return text, lp


def judge(text, dur, lp, lang_prob):
    drop, review = [], []
    if not text.strip():                drop.append("empty")
    if is_prompt_leak(text):            drop.append("prompt_leak")
    if has_loop(text):                  drop.append("repetition_loop")
    cps = len(text) / max(dur, 0.1)
    if cps > 28:                        review.append("too_many_chars")
    if cps < 2.0 and text.strip():      review.append("too_few_chars")
    if lp is not None and lp < -1.0:    review.append("low_confidence")
    if lang_prob is not None and lang_prob < 0.6: review.append("uncertain_language")
    return drop, review


def main():
    base = Path(BASE)
    src = base / "transcripts.jsonl"
    recs = [json.loads(l) for l in src.read_text(encoding="utf-8").splitlines() if l.strip()]
    redo_f = base / "transcripts_v2_redo.jsonl"
    done = {json.loads(l)["chunk_id"]: json.loads(l) for l in redo_f.read_text(encoding="utf-8").splitlines() if l.strip()} if redo_f.exists() else {}

    todo = []
    for r in recs:
        r["leak1"] = is_prompt_leak(r["text_raw"])
        if r["leak1"] or r["flags"] or has_loop(r["text_raw"]) or r["script"] in REDO_SCRIPTS:
            todo.append(r)
    print(f"pass-1 clips: {len(recs)} | prompt-leak suspects: {sum(r['leak1'] for r in recs)} | to re-check: {len(todo)}")
    if TEST_N:
        todo = random.Random(1).sample(todo, min(TEST_N, len(todo)))
    todo = [r for r in todo if r["chunk_id"] not in done]
    print(f"{len(done)} already re-checked | {len(todo)} to do")

    if todo:
        fw = _need("faster_whisper", "faster-whisper")
        try:
            import torch
            dev = "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            dev = "cpu"
        model = fw.WhisperModel(MODEL, device=dev, compute_type="float16" if dev == "cuda" else "int8")
        t0, fails = time.time(), 0
        with open(redo_f, "a", encoding="utf-8") as fo:
            for i, r in enumerate(todo, 1):
                try:
                    audio, dur = load_16k(r["path"])
                    lang, prob = detect(model, audio)
                    use = "en" if (lang == "en" and prob >= 0.8) else ("hi" if (lang in HI_LIKE and prob >= 0.8) else None)
                    text, lp = transcribe_clean(model, audio, use)
                    drop, review = judge(text, dur, lp, prob)
                    rec = {"chunk_id": r["chunk_id"], "text_pass1": r["text_raw"], "text_raw": text,
                           "lang_detected": lang, "lang_prob": round(prob, 2), "lang_used": use or "auto",
                           "avg_logprob": None if lp is None else round(lp, 3), "script": script_mix(text),
                           "drop_reasons": drop, "review_reasons": review}
                    fo.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    fails = 0
                except Exception as ex:
                    print(f"   [skipped] {r['chunk_id']}: {str(ex)[:120]}")
                    fails += 1
                    if fails >= 5:
                        raise SystemExit("5 clips failed in a row - stopping. Send me the error above.")
                if i % 50 == 0:
                    fo.flush()
                    el = time.time() - t0
                    print(f"[{i}/{len(todo)}] {el/60:.1f} min elapsed, ~{el/i*(len(todo)-i)/60:.0f} min left")
        done = {json.loads(l)["chunk_id"]: json.loads(l) for l in redo_f.read_text(encoding="utf-8").splitlines() if l.strip()}

    # merge: re-checked clips use the new result; untouched clips keep pass-1 text (status ok)
    out = []
    for r in recs:
        n = done.get(r["chunk_id"])
        if n:
            r2 = {**r, **n}
        else:
            r2 = {**r, "text_pass1": r["text_raw"], "lang_detected": None, "lang_prob": None, "lang_used": "hi(pass1)",
                  "drop_reasons": [], "review_reasons": []}
            if r["leak1"] or r["flags"]:
                r2["drop_reasons"] = ["not_rechecked"] if not TEST_N else []
        r2["status"] = "drop" if r2["drop_reasons"] else ("review" if r2["review_reasons"] else "ok")
        r2.pop("flags", None)
        out.append(r2)
    (base / "transcripts_v2.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in out), encoding="utf-8")

    try:
        import pandas as pd
        d = pd.DataFrame(out)
        print("\n================ QC SUMMARY ================")
        print(d.status.value_counts().to_string())
        print("\nstatus x voice:"); print(pd.crosstab(d.speaker, d.status))
        print("\nHours by voice x script (status=ok only):")
        ok = d[d.status == "ok"]
        print((ok.groupby(["speaker", "script"]).duration.sum() / 3600).round(2).unstack(fill_value=0))
        print("\nTotal ok hours:", round(ok.duration.sum() / 3600, 2), "| review hours:", round(d[d.status == 'review'].duration.sum() / 3600, 2))
        print("drop reasons:", dict(Counter(x for L in d.drop_reasons for x in L)))
        print("language detected for re-checked clips:", dict(Counter(x for x in d.lang_detected if x)))
        ch = d[(d.lang_detected == "en") & (d.text_pass1.str.contains(r"[\u0900-\u097F]"))]
        print(f"English clips that pass 1 had written in Devanagari: {len(ch)}")
        print("\nExamples of fixes:")
        for _, r in ch.head(5).iterrows():
            print(" before:", r.text_pass1[:90]); print(" after :", r.text_raw[:90])
    except Exception as ex:
        print("[summary skipped]", ex)


try:
    main()
except SystemExit as e:
    print(e)
except Exception:
    print("UNEXPECTED ERROR (send this to me):")
    traceback.print_exc()

pass-1 clips: 4324 | prompt-leak suspects: 889 | to re-check: 1553
0 already re-checked | 1553 to do
[50/1553] 1.1 min elapsed, ~33 min left
[100/1553] 2.2 min elapsed, ~32 min left
[150/1553] 3.3 min elapsed, ~31 min left
[200/1553] 4.3 min elapsed, ~29 min left
[250/1553] 5.4 min elapsed, ~28 min left
[300/1553] 6.4 min elapsed, ~27 min left
[350/1553] 7.4 min elapsed, ~26 min left
[400/1553] 8.5 min elapsed, ~24 min left
[450/1553] 9.7 min elapsed, ~24 min left
[500/1553] 10.7 min elapsed, ~23 min left
[550/1553] 11.8 min elapsed, ~21 min left
[600/1553] 12.8 min elapsed, ~20 min left
[650/1553] 13.9 min elapsed, ~19 min left
[700/1553] 15.0 min elapsed, ~18 min left
[750/1553] 16.1 min elapsed, ~17 min left
[800/1553] 17.3 min elapsed, ~16 min left
[850/1553] 18.8 min elapsed, ~16 min left
[900/1553] 20.7 min elapsed, ~15 min left
[950/1553] 21.9 min elapsed, ~14 min left
[1000/1553] 23.6 min elapsed, ~13 min left
[1050/1553] 25.0 min elapsed, ~12 min left
[1100/1553] 26.3 min elap